# Notebook 18 — VoxIntel-R Final Reliability and Decision Layer (Dual-Dataset)

**Status:** Final notebook of the current VoxIntel-R research phase (no Notebook 19 planned).

**Project:** VoxIntel-R — reference-free reliability estimation for ASR-to-intent voice systems.

**Position in the project:**

| Notebook | Role |
|---|---|
| 01–15 | Historical foundation: reference-aware diagnosis of how ASR errors propagate into intent failures |
| 16 | Reference-free VoxIntel-R architecture established |
| 17 | Cross-dataset complementarity test (**H1**), model-family selection. **H1 was NOT SUPPORTED.** |
| **18 (this notebook)** | Calibration (**H2**), selective prediction (**H3**), cost-sensitive decision making (**H4**) — tested under **each dataset's own artifact contract**, SLURP first, then FSC, then compared |

This notebook does not redesign VoxIntel-R, does not retrain the underlying Wav2Vec2 ASR model or the
DistilBERT intent model, and does not merge FSC and SLURP into one training/evaluation set. It reuses the
frozen per-utterance reference-free feature/prediction tables produced upstream (one per dataset).

**FSC and SLURP do NOT go through an identical pipeline in this version, by design:**

- **SLURP** (`voxintel_r_predictions.csv`) is an **existing evaluation population** with no independent
  validation/train split. This notebook evaluates its existing VoxIntel-R risk score as-is. It does
  **not** fit a new risk model, does **not** fit sigmoid/isotonic calibration, and does **not** grid-search
  a cost-optimal threshold on SLURP, because doing any of those things would require selecting a
  parameter using the same labels the result is then reported against — every one of those steps needs a
  held-out split SLURP's artifact does not provide.
- **FSC** (`fsc_voxintel_r_features.csv`) has a genuine `validation` (3,118 rows) and `test` (3,793 rows)
  population. This notebook fits the Random Forest risk models and every downstream calibrator/threshold
  **on FSC validation only**, freezes them, and evaluates **once** on FSC test.

**What changed from the first version of this notebook (v1 -> v2):** v1 assumed a single
`R[dataset]["y_val"]` / `R[dataset]["val_scores"]` / `R[dataset]["test_scores"]` shape existed for *every*
active dataset, including SLURP, and it assumed a `voxintel_r_risk` column existed in the FSC **feature**
artifact. Both assumptions are false and caused hard failures (`R["slurp"]["y_val"] is None`,
`fsc_voxintel_r_features.csv` has no `voxintel_r_risk` column). v2 removed both assumptions:
dataset-specific branching is explicit everywhere (`if evaluation_mode == "split": ... elif
evaluation_mode == "existing_evaluation_population": ...`), `R["slurp"]` and `R["fsc"]` store only the
fields that are actually valid for that dataset's protocol, and every artifact is schema-checked before
use rather than assumed to match what an earlier version of this notebook expected.

**What changed in this revision (v2 -> v3), from a methodology review of v2:**

1. **FSC calibration no longer fits on in-sample validation predictions.** Section 19 now produces
   **out-of-fold (OOF)** validation probabilities via 5-fold CV (each row scored by a model that never
   saw it during training); Sections 21-22 fit sigmoid/isotonic calibration on those OOF probabilities,
   not on the in-sample `predict_proba` of a model fit on all of validation. Fitting a calibrator on a
   model's own in-sample predictions is optimistic, because the model has partly memorized those rows.
2. **Calibration-variant selection no longer touches FSC test.** Section 23 now selects `raw` / `sigmoid`
   / `isotonic` for model C by lowest ECE **on the OOF validation predictions only**; v2 sorted on FSC-test
   ECE, which is test-set leakage into a modeling decision. Section 23b still reports full test-set
   calibration metrics for transparency, but that report can no longer change the selection.
3. **H4 defaults to `INCONCLUSIVE` under simulated severity.** FSC's cost-sensitivity analysis
   (Sections 28-30) assigns severity via deterministic hash bucketing unless a real per-intent mapping is
   supplied in `SEVERITY_OVERRIDES`. v2 could report `H4 = SUPPORTED` purely from wins against
   hash-assigned costs, which is not a valid empirical test of a cost-sensitivity hypothesis. v3 keeps the
   cost-sensitivity numbers as an explicitly-labeled sensitivity analysis, but Section 33 reports `H4`
   as `INCONCLUSIVE` unless `severity_is_simulated` is `False`.
4. **SLURP's H3 significance test now bootstraps ΔAURC, not ΔROC-AUC.** H3 is a claim about
   risk-coverage/selective-prediction performance, and Section 14's point estimate already uses AURC;
   v2's Section 15 bootstrapped ROC-AUC instead, which answers a related but different question. Section
   15 now bootstraps the AURC delta directly, so the point estimate and the significance test agree on
   what they are measuring.

**What changed in this revision (v3 -> v4), from a further methodology review of v3:**

1. **FSC calibration is no longer selected by validation ECE at all -- sigmoid is now pre-specified.** v3's
   fix (item 2 above) selected `raw` / `sigmoid` / `isotonic` by lowest ECE on out-of-fold validation
   predictions, which fixed the in-sample-overfitting leak but not a second, subtler one: isotonic
   regression is flexible enough to reshape itself almost exactly around whatever data it is fit on, so it
   could still land a near-zero ECE *on the very OOF rows it was fit on* without that generalizing --
   concretely, isotonic "won" v3's validation comparison by a wide margin (ECE ~0) and then did **not** win
   on FSC test (test ECE 0.01105 vs. raw's 0.00816). v4 pre-specifies `sigmoid` as model C's calibration
   method (`FSC_PRIMARY_CALIBRATION_VARIANT`, Section 09) before any calibration metric is computed, rather
   than building a further nested/cross-fitted calibration-selection scheme on top of FSC's already-tiny
   failure population. Isotonic is still fit and fully reported (Sections 22, 23b) as a labeled sensitivity
   analysis, but no longer feeds any threshold, cost policy, or verdict. See Section 23 for the full
   rationale and the live diagnostic check.
2. **FSC's H3 verdict is no longer decided from a point estimate alone.** v3 added FSC's own risk-coverage
   evaluation (Section 27) but did not bootstrap it, so Section 33 could call `H3 = SUPPORTED` on FSC from
   margins as small as 0.000024 AURC on only 23 test positives -- a difference indistinguishable from noise.
   New **Section 27b** runs the same paired-bootstrap-ΔAURC machinery already used for SLURP (now shared via
   `run_aurc_bootstrap_suite` and `verdict_h3` in Section 12) on FSC test. The H3 decision rule (Section 33)
   now requires a bootstrap-significant, risk-favoring win on top of the point-estimate win for `SUPPORTED`;
   a point-estimate win with no significant comparison is reported as `INCONCLUSIVE` instead. The same
   stricter rule was applied to SLURP's H3 too, for consistency, and SLURP also gained an `always_execute`
   bootstrap comparison it previously lacked.
3. **H2's calibration-selection metric is now explicit.** Section 33's H2 criteria now state directly that
   ECE is the primary metric and MCE is a secondary reliability diagnostic, resolving the earlier tension
   where a metric (MCE) that was never part of the selection procedure was still part of the pass/fail
   bar -- moot for the *selection* question now that Section 23 no longer selects by any metric, but still
   worth stating explicitly since MCE is a real part of the H2 SUPPORTED/PARTIALLY SUPPORTED boundary.

## 01 — Executive summary

**Where the project stands going into this notebook:**

- Notebook 16 established a reference-free VoxIntel-R pipeline: ASR-native uncertainty features and
  intent-native uncertainty features, with no reference transcript, WER, CER, or ground-truth intent
  allowed as inference-time inputs.
- Notebook 17 tested **H1** — whether ASR-native uncertainty adds incremental predictive value beyond
  intent-native uncertainty for predicting downstream intent failure. On the frozen FSC test split,
  **H1 was NOT SUPPORTED**: the combined model (intent + ASR features) did not significantly outperform
  the intent-only model (ΔROC-AUC = −0.0247, 95% CI [−0.1441, +0.0926]). A provisional (non-frozen) SLURP
  comparison pointed the same direction (see Section 06).
- The interpretation carried forward is **not** "ASR uncertainty is useless" — it is that intent-native
  uncertainty may already compress most of the useful downstream-failure signal. **This notebook does not
  reopen H1** — it tests the three hypotheses that remain after H1 failed.

**What this notebook adds, run under each dataset's own artifact contract (SLURP evaluated first, then
FSC), then compared:**

1. **H2 — Calibration.** Are the model's predicted failure probabilities trustworthy as probabilities?
2. **H3 — Selective prediction.** Can VoxIntel-R risk decide, per-utterance, whether to execute or
   clarify — and does this beat naive confidence-threshold baselines?
3. **H4 — Cost-sensitive decision making.** Can a severity-aware policy reduce expected cost relative to
   a uniform threshold policy (severity tiers are *simulated*, not real-world ground truth)?

**What this notebook adds across datasets:** a final synthesis (Section 34) stating, for each of
H2/H3/H4, whether the FSC and SLURP verdicts **agree** (methodology generalizes), **disagree**
(dataset/domain-dependent), or **could not both be evaluated** (SLURP's artifact contract does not support
every FSC analysis — this is stated explicitly rather than papered over).

All FSC thresholds, calibration mappings, and cost policies below are fit on **FSC validation only** and
evaluated **once** on FSC test. SLURP's existing risk is evaluated directly against SLURP's own population;
nothing is fit on it. An explicit audit table (Section 08 / `AUDIT_LOG`) makes this contract auditable,
tagged by dataset and by whether a step involved fitting at all.

## 02 — What is VoxIntel-R?

**Core research question:**

> Can a voice system predict when its ASR-derived intent is likely to fail, using only information
> available at inference time, and use that calibrated risk to selectively execute or clarify the
> command — and does the answer hold across more than one dataset?

**Architecture:**

```
Audio
  |
  v
ASR (Wav2Vec2, frozen)
  |
  v
ASR hypothesis ---- ASR-native uncertainty (9 features)
  |
  v
Intent model (DistilBERT, frozen) ---- Intent uncertainty (3 features)
  |
  v
Reference-free risk model
  |     FSC:   fit per-dataset in this notebook (validation -> frozen -> test)
  |     SLURP: already fit upstream; this notebook evaluates it as-is
  v
Calibrated failure probability
  |
  v
Selective decision
  |-- EXECUTE
  `-- CLARIFY / ABSTAIN
```

**The inference-time contract (identical for both datasets):**

- Reference transcripts / ground-truth intent *may* be used to construct the `intent_failed` label and
  for offline subgroup analysis — never as an inference-time feature.
- Reference transcripts, WER, CER, reference-derived taxonomy, ground-truth intent, and any
  reference-alignment feature **must never** be used to construct an inference-time risk feature.
  Section 09/11 audit this explicitly for every artifact loaded in this notebook, on both datasets.

**Plain-English version:** a normal voice assistant does *hear → transcribe → understand → execute.*
VoxIntel-R inserts one more step: *hear → transcribe → understand → estimate how likely I am to be
wrong → decide whether to execute or ask for clarification.*

## 03 — VoxIntel v1 → VoxIntel-R evolution

| Stage | Question | Status |
|---|---|---|
| **VoxIntel v1** | Which ASR errors cause downstream intent failure? | Descriptive / diagnostic |
| **Notebook 15** | Can semantic/reference-aware error features predict intent failure? | Predictive, but reference-aware |
| **VoxIntel-R** | Can the system estimate when it should not trust itself, using only inference-time information, **on more than one dataset**? | Reference-free / operational / cross-dataset |

Notebook 15's semantic and taxonomy features depended on reference-aware, post-hoc analysis. That makes
them useful for *offline diagnosis*, but not deployable as a *real-time* reliability signal. VoxIntel-R
restricts itself to inference-time signals; this notebook additionally asks whether that restriction still
works once you leave the single dataset (FSC) it was originally validated on.

## 04 — Historical research foundation (Notebooks 01–15)

The tables below restate historical results from the project README. They are **not** recomputed in
this notebook — they are reference-aware, offline, diagnostic results that motivated VoxIntel-R, not
part of its frozen reference-free evaluation. They are prior evidence/context only.

### Notebook 08 — SLURP test-time comparison

| Transcript source | Accuracy | Macro-F1 | Top-3 |
|---|---|---|---|
| Ground truth | 0.8577 | 0.7032 | 0.9444 |
| Baseline ASR | 0.3849 | 0.3082 | 0.5157 |
| Fine-tuned ASR | 0.7343 | 0.5554 | 0.8454 |
| **Recovery (fine-tuned − baseline)** | **+0.3494** | **+0.2473** | — |

This is evidence that downstream accuracy recovers alongside improved ASR transcripts on SLURP — it is
**not** causal proof of a universal ASR-quality effect, and it predates the reference-free constraint.

### Notebook 14 — semantic-impact analysis

WER alone was shown to be an insufficient proxy for downstream semantic risk. Historical per-category
error-rate recovery (baseline → fine-tuned):

| Error category | Baseline | Fine-tuned | Reduction |
|---|---|---|---|
| number_error | ≈0.682 | ≈0.289 | ≈0.394 |
| proper_noun_error | ≈0.645 | ≈0.298 | ≈0.347 |
| deletion | ≈0.211 | ≈0.085 | ≈0.126 |
| entity_error | ≈0.093 | ≈0.043 | ≈0.050 |

### Notebook 15 — reference-aware predictive features

| Model | Baseline ASR ROC-AUC | Fine-tuned ASR ROC-AUC |
|---|---|---|
| Proxy-only logistic | ≈0.795 | ≈0.707 |
| Taxonomy-only logistic | — | ≈0.824 |
| Combined logistic | ≈0.841 | ≈0.841 |
| Combined logistic (CV) | ≈0.842 | ≈0.834 |

Paired bootstrap improvement of combined vs. proxy-only: baseline ASR ≈ **+0.046** (95% CI [0.033, 0.060]);
fine-tuned ASR ≈ **+0.134** (95% CI [0.109, 0.156]). These results were **reference-aware** — motivating,
but not solving, the operational reference-free problem that VoxIntel-R addresses.

### Notebook 16/17 — FSC intent-failure heterogeneity and model-family selection

Earlier FSC analysis showed downstream intent failure is not uniform across intents — fragile intents
included `query`, `music`, `quirky` (baseline failure rate ≈1.0), `social_query` (≈0.92), `alarm_remove`
(≈0.86), and `email_query` (≈0.85), among others. This motivates keeping subgroup/intent-level analysis
available rather than reporting only one aggregate metric (see Section 11 and the FSC test evaluation
below).

5-fold CV ROC-AUC by feature group on FSC validation (prior run, context only — not recomputed here):

| Feature group | Random Forest | Logistic Regression |
|---|---|---|
| A — ASR-only | ≈0.615 | ≈0.625 |
| B — Intent-only | ≈0.91 | ≈0.815 |
| C — Combined | ≈0.88 | ≈0.815 |

Prior Random Forest feature-importance ranking on combined features (context only): `intent_margin` >
`intent_confidence` > `intent_entropy` > `asr_median_confidence` > `asr_std_entropy` > `asr_max_entropy` >
`asr_mean_entropy` > `asr_mean_confidence` > `asr_std_confidence` > `asr_min_confidence` > `audio_duration`
> `asr_num_frames`. Intent-side features dominated. Random Forest was selected as the model family
(mean CV ROC-AUC 0.8438 ± 0.0499 vs. 0.7818 ± 0.0453 for logistic regression) and is reused for FSC's
Section 19 model fit below — the family choice itself is not re-derived here, only refit per this
notebook's own validation/test discipline.

**None of the numbers on this page are treated as this notebook's own results.** Section 19 onward
recomputes FSC's models from scratch on this run's data and reports actual, live numbers.

## 05 — Current state after Notebook 17

**Datasets used in this notebook, and how each is used (never pooled):**

| Dataset | Protocol | Utterances (expected) |
|---|---|---|
| FSC | validation → fit/freeze → test | validation: 3,118 &nbsp;/&nbsp; test: 3,793 |
| SLURP | existing evaluation population (no fitting) | 8,688 |

FSC's expected counts come from the frozen Notebook 16/17 artifact and are hard-checked in Section 11.
SLURP's expected count (8,688) comes from the existing `voxintel_r_predictions.csv` artifact and is also
hard-checked in Section 11 — but unlike FSC, this is a single population, not a validation/test pair.

**Corrected FSC ASR evaluation (Section 08 of the FSC pipeline, context only):**

| Split | WER | CER |
|---|---|---|
| Validation | 0.032715 (≈3.27%) | 0.027346 (≈2.73%) |
| Test | 0.017975 (≈1.80%) | 0.018359 (≈1.84%) |

An earlier implementation had incorrectly reported ≈99% WER / ≈82% CER. That was an evaluation and
normalization bug (case sensitivity and whitespace handling), **not** a broken ASR model. **The Wav2Vec2
FSC checkpoint is not retrained in this notebook**, and no ASR model is retrained for SLURP here either —
this notebook only ever (a) fits the lightweight tabular risk layer for FSC, on FSC validation only, or
(b) evaluates SLURP's already-existing risk score. It never touches raw audio or reference transcripts.

**Downstream intent failure (the binary risk target, same definition on both datasets):**

```
intent_failed = (predicted_intent_id != ground_truth_intent_id)
```

| Dataset | Population | Failures | Rate (frozen/expected) |
|---|---|---|---|
| FSC | Validation | 93 / 3,118 | 2.9827% |
| FSC | Test | 23 / 3,793 | 0.6064% |
| SLURP | Evaluation population | 1,851 / 8,688 | 21.3052% |

The ground-truth intent is used **only** to construct this target and for subgroup analysis — never as an
inference-time feature, on either dataset. Section 11 recomputes these counts live from the loaded
artifacts and hard-fails if they disagree with the frozen expectations above.

**Reference-free feature schema (12 features, identical contract for both datasets):**

| Group | Features |
|---|---|
| A — ASR-native (9) | `asr_mean_confidence`, `asr_min_confidence`, `asr_std_confidence`, `asr_median_confidence`, `asr_mean_entropy`, `asr_max_entropy`, `asr_std_entropy`, `asr_num_frames`, `audio_duration` |
| B — Intent-native (3) | `intent_confidence`, `intent_entropy`, `intent_margin` |
| C — Combined (12) | A ∪ B |

**Forbidden inference-time features (audited in Section 09/11, both datasets):** reference transcript,
WER, CER, reference-derived taxonomy, ground-truth intent, lexical reference overlap, semantic reference
alignment.

**What is genuinely new vs. already existing, per dataset:**

| Dataset | Risk model | Calibration | Selective thresholds | Cost-optimal policy |
|---|---|---|---|---|
| FSC | fit in Section 19 (this notebook) | fit in Sections 21–22, pre-specified as sigmoid in Section 23, reported on test in 23b (this notebook) | fit in Section 26 (this notebook) | fit in Section 30 (this notebook) |
| SLURP | already exists upstream — evaluated only | evaluated only (no new fit) | descriptive only (no held-out split to fit a threshold on) | **not evaluated** — no held-out split to select a cost-optimal threshold without leakage |

## 06 — H1 and why it failed

**H1 (already evaluated in Notebook 17, not rerun here):** ASR-native uncertainty provides incremental
predictive value beyond intent-native uncertainty (model C vs. model B).

**Frozen FSC test result:**

| Metric | Δ (C − B) | 95% CI | Significant? |
|---|---|---|---|
| ROC-AUC | −0.0247 | [−0.1441, +0.0926] | No |
| PR-AUC | −0.0398 | [−0.1453, +0.0531] | No |
| Brier | +0.0005 | [−0.0005, +0.0014] | No |

**Verdict: H1 is NOT SUPPORTED on FSC.**

This is not evidence that "ASR uncertainty is useless" — model C's Random Forest still assigns
substantial feature importance to ASR-native features (see Section 04). Provisional (not frozen) SLURP
numbers from Notebook 17 pointed the same direction (combined did not beat intent-only). **This notebook
does not re-run H1 on either dataset** — H1 is a frozen, inherited Notebook 17 result. Sections 12–27
below instead give SLURP its own evidence-appropriate H2/H3/H4 treatment (evaluation-only, using SLURP's
existing frozen risk prediction) and give FSC its full validation → frozen model → held-out test
treatment, then compare what survives.

**Why H1 likely failed — technical explanation (context, not re-derived here):**

1. FSC ASR is already strong (test WER ≈ 1.80%).
2. Downstream intent failures are rare on FSC (23 / 3,793 = 0.6064%), which widens confidence intervals.
3. Intent uncertainty is *downstream* of the ASR hypothesis and may already encode much of the semantic
   damage an imperfect ASR output causes.
4. ASR uncertainty may therefore be partially redundant with intent uncertainty.
5. With few positive test cases, confidence intervals are wide on both datasets — treat any single
   significant delta with appropriate caution.

## 07 — Research hypotheses for this notebook

These are the three hypotheses that remain after H1 failed, carried forward from Notebook 17/prior
project planning with their canonical wording preserved:

**H2 — Calibration.** A reference-free VoxIntel-R failure estimator can produce sufficiently calibrated
downstream intent-failure probabilities to support reliable risk-based selective decisions. Evaluated
independently on FSC and SLURP.

**H3 — Selective prediction.** Calibrated reference-free failure risk can improve the risk-coverage
trade-off over naive single-signal confidence-threshold baselines (always-execute, ASR-confidence
threshold, intent-confidence threshold). Evaluated independently on FSC and SLURP.

**H4 — Cost-sensitive selective decision making.** A severity/cost-aware selective policy can reduce
expected decision cost compared with uniform confidence-threshold policies. Severity tiers are
**simulated / assigned** on both datasets, not real-world ground truth. Evaluated on FSC; **not**
evaluated on SLURP (Section 16 explains why — SLURP has no held-out split to select a cost-optimal
threshold on without evaluating it against the same labels used to pick it). **On FSC, H4 itself defaults
to `INCONCLUSIVE`, reported as a sensitivity analysis only, unless a real per-intent severity mapping is
supplied** (Section 28) — a win-count under hash-assigned severity is not a valid test of a cost-sensitivity
hypothesis, only a demonstration of how the policy behaves under an assumed cost structure.

**H2x/H3x/H4x — Cross-dataset generalization.** For each of H2, H3, and H4: do the FSC and SLURP verdicts
agree, where both could be evaluated? Section 34 states this explicitly, using only the two datasets'
already-frozen, independently-computed verdicts — it never re-fits anything on pooled data to answer this.

**Order of operations in this notebook: SLURP is evaluated first (Sections 13–18), then FSC goes through
its full validation → frozen model → test protocol (Sections 19–32), then the two are compared
(Sections 33–34).** This mirrors the artifact contract: SLURP's evaluation does not depend on anything
fit in the FSC arm, and vice versa.

Decision criteria for each per-dataset verdict (SUPPORTED / PARTIALLY SUPPORTED / NOT SUPPORTED /
INCONCLUSIVE) are stated in Section 33, before results are read, to avoid post-hoc rationalization.

## 08 — Experimental contract and leakage rules

| Rule | Enforcement in this notebook |
|---|---|
| No retraining Wav2Vec2 / DistilBERT | Only the lightweight tabular risk model (Random Forest over the 12-feature schema) is fit in this notebook, and only for FSC, on FSC's own validation split. SLURP's risk is read from its existing artifact, never fit. |
| No reference transcript, WER, CER, taxonomy, or ground-truth intent as an inference-time feature | Enforced programmatically in Section 11 (forbidden-feature audit) and Section 09 (`ALLOWED_FEATURE_COLUMNS`), for both datasets. |
| FSC calibration fit on validation only, evaluated on test | Sections 19, 21–23b, FSC only. |
| FSC calibration method is pre-specified, not chosen by whichever metric scores best | `FSC_PRIMARY_CALIBRATION_VARIANT = "sigmoid"` (Section 09); Section 23 fixes `best_variant_C` to this constant rather than sorting a metrics table. Isotonic is fit and reported (Sections 22, 23b) but never used downstream. |
| FSC selective thresholds chosen on validation only | Section 26, FSC only. |
| H3 "beats" a baseline only if the win survives a bootstrap, for both datasets | `verdict_h3` (Section 12) requires the AURC bootstrap CI (Section 15 for SLURP, Section 27b for FSC) to exclude zero and favor the risk score, not just a lower point-estimate AURC; otherwise the verdict is `INCONCLUSIVE` rather than `SUPPORTED`. |
| FSC cost policies chosen on validation only | Section 30, FSC only. |
| SLURP has no validation/train split | `DATASET_CONFIG["slurp"]["evaluation_mode"] == "existing_evaluation_population"`; `R["slurp"]` never contains `y_val`, `val_scores`, or a fitted calibrator/threshold. Any step in this notebook that would need one explicitly skips SLURP and logs why in `AUDIT_LOG`, rather than fabricating a split. |
| Test labels never used to tune anything (FSC) | Every "fit"/"select" call for FSC takes a `val_df` scoped to FSC; every "evaluate" call that touches FSC `test_df` happens only after the corresponding fit/select step, and is not looped back into any decision. |
| **FSC and SLURP never pooled** | No model, calibrator, threshold, or cost policy is ever fit on rows from both datasets combined. Section 34 compares already-computed per-dataset verdicts; it does not touch either dataset's rows again. |
| Simulated severity explicitly labeled | Every cost-model table and plot (FSC only) is titled/logged as `SIMULATED / ASSIGNED COST ASSUMPTIONS`. |
| Every artifact schema-checked before use | Section 10's loader inspects actual columns present (`_has_required_columns`) before trusting any file, for both datasets, and raises a explicit, informative error rather than silently falling back to another dataset's file. |

Section 09 prints a machine-readable audit table (`AUDIT_LOG`) that is appended to throughout the
notebook and saved as part of the final artifact manifest. Every entry is tagged with `dataset` (`fsc` or
`slurp`) and whether the step involved fitting parameters (`fit_on != "-"`) or was evaluation-only.

## 09 — Reproducibility, paths, and configuration

All paths and dataset contracts are centralized in the next two cells as `Path` objects and a
`DATASET_CONFIG` dict, so nothing is scattered through the notebook as a string literal. Edit **only
these two cells** to point the notebook at a different machine/repo layout.

**Artifact contract used below (per the current project state):**

- `FSC_FEATURES_ARTIFACT` (`reports/fsc_voxintel_r_features.csv`) — 6,911 rows × 14 columns: the 12
  reference-free features, `intent_failed`, and `split` (`validation`/`test`). It does **not** contain a
  pre-computed risk column — this notebook fits that risk model itself, in Section 19, on FSC validation
  only.
- `SLURP_PREDICTIONS_ARTIFACT` (`reports/voxintel_r_predictions.csv`) — an existing evaluation population
  (expected 8,688 rows) that **does** contain an existing VoxIntel-R risk column. This notebook evaluates
  that risk directly; it does not refit anything on SLURP.

In [1]:
import sys, platform, json, warnings, hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
try:
    import sklearn
    print("scikit-learn:", sklearn.__version__)
except ImportError:
    print("scikit-learn: NOT INSTALLED")
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
try:
    import torch
    print("PyTorch     : not imported in this notebook (no ASR/intent retraining occurs)")
except ImportError:
    print("PyTorch     : not imported in this notebook (no ASR/intent retraining occurs)")
try:
    import transformers
    print("Transformers: not imported in this notebook (no ASR/intent retraining occurs)")
except ImportError:
    print("Transformers: not imported in this notebook (no ASR/intent retraining occurs)")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Python      : 3.10.19
Platform    : Windows-10-10.0.26200-SP0
scikit-learn: 1.7.2
numpy       : 1.26.4
pandas      : 2.3.3
PyTorch     : not imported in this notebook (no ASR/intent retraining occurs)


c:\Users\ACER\anaconda3\envs\torch26\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers: not imported in this notebook (no ASR/intent retraining occurs)


In [2]:
# ---------------------------------------------------------------------
# Paths -- edit this cell only to point at a different machine/layout
# ---------------------------------------------------------------------
REPO_ROOT = Path(r"C:\Users\ACER\OneDrive\Desktop\VoxIntel")

DATA_DIR = REPO_ROOT / "data" / "raw" / "fsc" / "fluent_speech_commands_dataset"
FSC_AUDIO_DIR = DATA_DIR / "wavs"

SLURP_DATA_DIR = REPO_ROOT / "data" / "raw" / "slurp" / "dataset" / "slurp"

MODELS_DIR = REPO_ROOT / "models"
FSC_ASR_MODEL_DIR = MODELS_DIR / "wav2vec2_fsc"
SRC_DIR = REPO_ROOT / "src"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
REPORTS_DIR = REPO_ROOT / "reports"

OUT_DIR = REPORTS_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Primary artifact contract (Section 09). Do not point these at a different
# dataset's file -- Section 10's loader will refuse to substitute one for
# the other even if you do.
FSC_FEATURES_ARTIFACT = REPORTS_DIR / "fsc_voxintel_r_features.csv"
SLURP_PREDICTIONS_ARTIFACT = REPORTS_DIR / "voxintel_r_predictions.csv"

# Kept for readability elsewhere in the notebook / for anyone grepping for
# the old names used in Notebooks 16-17.
FSC_FEATURE_IMPORTANCE_CSV = REPORTS_DIR / "fsc_voxintel_r_feature_importance.csv"

COLUMN_MAP = {
    "split": "split",
    "target": "intent_failed",
    "id": "utterance_id",
    "intent_label": None,  # auto-detected in Sections 17 (SLURP) / 28 (FSC) if a real column exists
}

VALIDATION_SPLIT_VALUES = {"validation", "val", "dev", "development"}
TEST_SPLIT_VALUES = {"test"}

ASR_FEATURES = [
    "asr_mean_confidence",
    "asr_min_confidence",
    "asr_std_confidence",
    "asr_median_confidence",
    "asr_mean_entropy",
    "asr_max_entropy",
    "asr_std_entropy",
    "asr_num_frames",
    "audio_duration",
]

INTENT_FEATURES = [
    "intent_confidence",
    "intent_entropy",
    "intent_margin",
]

COMBINED_FEATURES = ASR_FEATURES + INTENT_FEATURES
ALLOWED_FEATURE_COLUMNS = set(COMBINED_FEATURES)

FORBIDDEN_FEATURE_SUBSTRINGS = [
    "reference",
    "ref_transcript",
    "wer",
    "cer",
    "taxonomy",
    "ground_truth_intent",
    "gt_intent",
    "lexical_overlap",
    "semantic_align",
    "ref_",
    "reference_transcript",
]

# Candidate column names for SLURP's *existing* VoxIntel-R risk score.
# The first one found in the artifact is used; none of these are assumed
# to exist without checking (Section 10 verifies before using any of them).
SLURP_RISK_COLUMN_CANDIDATES = [
    "voxintel_r_risk",
    "existing_voxintel_r_risk",
    "risk_score",
    "prob_failed",
    "prob_failed_C_combined",
]

# Optional per-feature-group prediction columns that MAY exist in an
# artifact (FSC or SLURP). Only used if actually present -- never assumed.
GROUP_SCORE_COLUMN_CANDIDATES = {
    "A": ["prob_failed_A_asr_only", "prob_failed_A"],
    "B": ["prob_failed_B_intent_only", "prob_failed_B"],
    "C": ["prob_failed_C_combined", "prob_failed_C", "voxintel_r_risk"],
}

DATASETS = ["slurp", "fsc"]  # SLURP evaluated first, then FSC (Section 07)

DATASET_CONFIG = {
    "fsc": {
        "label": "FSC",
        "features_csv": FSC_FEATURES_ARTIFACT,
        "evaluation_mode": "split",
        "split_required": True,
        "expected_val_n": 3118,
        "expected_test_n": 3793,
        "expected_val_failures": 93,
        "expected_test_failures": 23,
        # FSC's feature artifact has NO pre-existing risk column -- the risk
        # model is fit by this notebook (Section 19). Do not require one at
        # load time.
        "risk_column_required_at_load": False,
        "risk_column_candidates": [],
    },
    "slurp": {
        "label": "SLURP",
        "features_csv": SLURP_PREDICTIONS_ARTIFACT,
        "evaluation_mode": "existing_evaluation_population",
        "split_required": False,
        "expected_eval_n": 8688,
        "expected_val_n": None,
        "expected_test_n": None,
        "expected_val_failures": None,
        "expected_test_failures": None,
        # SLURP's artifact IS expected to already contain a risk column --
        # this is the "existing VoxIntel-R risk" this notebook evaluates.
        "risk_column_required_at_load": True,
        "risk_column_candidates": SLURP_RISK_COLUMN_CANDIDATES,
    },
}

TARGET_COLUMN = COLUMN_MAP["target"]
TARGET_COVERAGE = 0.90

# Pre-specified (not data-driven) calibration method for FSC model C. Sigmoid/Platt calibration is
# parametric and monotonic, so it cannot reshape itself around its own fitting data the way isotonic
# regression can -- see Section 23 for the diagnostic evidence behind this choice. Isotonic is still
# fit and reported everywhere sigmoid is (Sections 22-23b), but only as a sensitivity analysis: it is
# never used for thresholds, cost policies, or the primary H2/H3 verdicts.
FSC_PRIMARY_CALIBRATION_VARIANT = "sigmoid"

AUDIT_LOG = []

def audit(dataset, stage, fit_on, frozen, evaluated_on, note=""):
    entry = {
        "dataset": dataset,
        "stage": stage,
        "fit_on": fit_on,
        "frozen": frozen,
        "evaluated_on": evaluated_on,
        "note": note,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    AUDIT_LOG.append(entry)
    return entry

audit(
    "-", "notebook_start", fit_on="-",
    frozen="Wav2Vec2 frozen; DistilBERT frozen; no ASR/intent retraining",
    evaluated_on="-",
    note="Notebook 18 consumes existing VoxIntel-R artifacts. Order: " + ", ".join(DATASETS),
)

print("=" * 80)
print("NOTEBOOK 18 -- DATASET CONFIGURATION")
print("=" * 80)
print()
print("REPO_ROOT:", REPO_ROOT)
print("REPORTS_DIR:", REPORTS_DIR)
print("DATASETS (evaluation order):", DATASETS)
print()

for ds in DATASETS:
    cfg = DATASET_CONFIG[ds]
    print(f"[{ds.upper()}]")
    print(f"  Label:                    {cfg['label']}")
    print(f"  Features/predictions CSV: {cfg['features_csv']}")
    print(f"  Evaluation mode:          {cfg['evaluation_mode']}")
    print(f"  Split required:           {cfg['split_required']}")
    print(f"  Risk column required:     {cfg['risk_column_required_at_load']}")
    if cfg["risk_column_candidates"]:
        print(f"  Risk column candidates:   {cfg['risk_column_candidates']}")
    if cfg["evaluation_mode"] == "split":
        print(f"  Expected validation:      {cfg['expected_val_n']}")
        print(f"  Expected test:            {cfg['expected_test_n']}")
    else:
        print(f"  Expected evaluation pop.: {cfg['expected_eval_n']}")
    print()

print("Reference-free feature count:", len(COMBINED_FEATURES))
print("Reference-free features:")
for feature in COMBINED_FEATURES:
    print(" ", feature)
print()
print("Target column:", TARGET_COLUMN)
print("Random seed:", RANDOM_SEED)
print()
print("No ASR/intent model retraining occurs in Notebook 18.")

NOTEBOOK 18 -- DATASET CONFIGURATION

REPO_ROOT: C:\Users\ACER\OneDrive\Desktop\VoxIntel
REPORTS_DIR: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports
DATASETS (evaluation order): ['slurp', 'fsc']

[SLURP]
  Label:                    SLURP
  Features/predictions CSV: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\voxintel_r_predictions.csv
  Evaluation mode:          existing_evaluation_population
  Split required:           False
  Risk column required:     True
  Risk column candidates:   ['voxintel_r_risk', 'existing_voxintel_r_risk', 'risk_score', 'prob_failed', 'prob_failed_C_combined']
  Expected evaluation pop.: 8688

[FSC]
  Label:                    FSC
  Features/predictions CSV: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_features.csv
  Evaluation mode:          split
  Split required:           True
  Risk column required:     False
  Expected validation:      3118
  Expected test:            3793

Reference-free feature count: 12
Reference-free features:

## 10 — Load VoxIntel-R artifacts (per dataset, schema-checked)

This section loads each dataset's artifact using **that dataset's own contract** from `DATASET_CONFIG`.
It never lets one dataset's file stand in for another's. Key differences from the FSC-only pipeline this
project started with:

- **FSC** is only required to have the 12 reference-free features, `intent_failed`, and a split column.
  It is **not** required to already contain a risk column -- Section 19 fits that.
- **SLURP** *is* required to already contain a risk column (one of `SLURP_RISK_COLUMN_CANDIDATES`), and is
  **not** required to have a split column, because it is evaluated as a single population.

If a required column is missing, the loader raises an explicit `FileNotFoundError`/`ValueError` naming
exactly what was expected and what was found -- it never silently falls back to another dataset's file or
fabricates a split.

In [3]:
def _read_header(path: Path):
    try:
        return set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return None


def _resolve_risk_column(cols: set, dataset: str):
    '''Return the first present candidate risk column, or None if none is present.'''
    cfg = DATASET_CONFIG[dataset]
    for candidate in cfg["risk_column_candidates"]:
        if candidate in cols:
            return candidate
    return None


def _has_required_columns(path: Path, dataset: str) -> bool:
    cols = _read_header(path)
    if cols is None:
        return False

    cfg = DATASET_CONFIG[dataset]

    if not set(COMBINED_FEATURES).issubset(cols):
        return False

    if TARGET_COLUMN not in cols:
        return False

    if cfg["risk_column_required_at_load"]:
        if _resolve_risk_column(cols, dataset) is None:
            return False

    if cfg["split_required"]:
        has_split = any(
            (c == COLUMN_MAP["split"] or "split" in c.lower() or "subset" in c.lower())
            for c in cols
        )
        if not has_split:
            return False

    return True


def resolve_artifact_csv(dataset: str) -> Path:
    if dataset not in DATASET_CONFIG:
        raise KeyError(f"Unknown dataset: {dataset}")

    cfg = DATASET_CONFIG[dataset]
    primary = Path(cfg["features_csv"])
    candidates = [primary]

    if primary.exists() and _has_required_columns(primary, dataset):
        return primary

    # Only ever search for *this* dataset's own files (filename must start
    # with the dataset name) -- never substitute the other dataset's file.
    if REPORTS_DIR.exists():
        for p in sorted(REPORTS_DIR.glob(f"{dataset}_*.csv")):
            if p == primary:
                continue
            candidates.append(p)
            if _has_required_columns(p, dataset):
                return p

    risk_note = (
        f"one of {cfg['risk_column_candidates']}"
        if cfg["risk_column_required_at_load"]
        else "not required at load time (fit by this notebook instead, if applicable)"
    )
    split_note = "A split column is REQUIRED." if cfg["split_required"] else \
                 "A split column is NOT required (existing evaluation population)."

    raise FileNotFoundError(
        f"\n[{dataset}] Could not find a valid VoxIntel-R artifact.\n\n"
        f"Expected primary path:\n  {primary}\n\n"
        f"Required reference-free features ({len(COMBINED_FEATURES)}):\n"
        f"  {sorted(COMBINED_FEATURES)}\n\n"
        f"Required target:\n  {TARGET_COLUMN}\n\n"
        f"Required risk column:\n  {risk_note}\n\n"
        f"Split contract:\n  {split_note}\n\n"
        f"Candidates checked:\n  {[str(c) for c in candidates]}\n\n"
        f"This notebook will NOT:\n"
        f"  - substitute another dataset's artifact\n"
        f"  - randomly split an existing evaluation population\n"
        f"  - derive features from raw audio/transcripts\n"
        f"  - invent a risk column that is not actually in the file\n"
    )


def _forbidden_feature_audit():
    hits = []
    for feature in COMBINED_FEATURES:
        lf = feature.lower()
        for forbidden in FORBIDDEN_FEATURE_SUBSTRINGS:
            if forbidden in lf:
                hits.append((feature, forbidden))
    return hits


def load_dataset(dataset: str) -> dict:
    cfg = DATASET_CONFIG[dataset]
    resolved_path = resolve_artifact_csv(dataset)

    print()
    print("=" * 80)
    print(f"LOADING {dataset.upper()} VOXINTEL-R ARTIFACT")
    print("=" * 80)
    print("\nPath:", resolved_path)

    raw_df = pd.read_csv(resolved_path)
    print("Rows:", len(raw_df))
    print("Columns:", len(raw_df.columns))

    missing_features = [f for f in COMBINED_FEATURES if f not in raw_df.columns]
    if missing_features:
        raise ValueError(f"[{dataset}] Missing required reference-free features:\n{missing_features}")

    if TARGET_COLUMN not in raw_df.columns:
        raise ValueError(f"[{dataset}] Missing target column: {TARGET_COLUMN}")

    # Forbidden-token audit is against the REQUIRED feature schema (should
    # always pass -- this is a defensive check against config drift) and,
    # separately, against every column actually present in the raw file, so
    # a forbidden-looking column silently sitting in the CSV cannot later be
    # picked up by accident.
    schema_hits = _forbidden_feature_audit()
    if schema_hits:
        raise ValueError(
            f"[{dataset}] Forbidden content detected in the reference-free feature "
            f"schema itself (config error): {schema_hits}"
        )

    present_forbidden = [
        c for c in raw_df.columns
        if any(tok in c.lower() for tok in FORBIDDEN_FEATURE_SUBSTRINGS)
        and c not in ALLOWED_FEATURE_COLUMNS
    ]

    risk_column = _resolve_risk_column(set(raw_df.columns), dataset)
    if cfg["risk_column_required_at_load"] and risk_column is None:
        raise ValueError(
            f"[{dataset}] None of the expected risk columns "
            f"{cfg['risk_column_candidates']} were found in {resolved_path.name}. "
            f"Columns present: {sorted(raw_df.columns)}"
        )

    # Any group-level (A/B/C) prediction columns actually present -- used
    # opportunistically later (e.g. SLURP H2-style A-vs-B checks) and never
    # assumed if absent.
    available_group_scores = {}
    for group, candidates_ in GROUP_SCORE_COLUMN_CANDIDATES.items():
        for c in candidates_:
            if c in raw_df.columns:
                available_group_scores[group] = c
                break

    print("\nReference-free feature schema:  PASS")
    print("Target column:                  ", TARGET_COLUMN)
    print("Resolved risk column:           ", risk_column)
    print("Forbidden-looking columns present (not used as features):", present_forbidden or "none")
    print("Available group-level score columns (opportunistic, may be empty):", available_group_scores or "none")

    evaluation_mode = cfg["evaluation_mode"]

    if evaluation_mode == "existing_evaluation_population":
        eval_df = raw_df.copy().reset_index(drop=True)
        expected_eval_n = cfg.get("expected_eval_n")

        print("\nEvaluation mode:", evaluation_mode)
        print("Evaluation rows:", len(eval_df))
        if expected_eval_n is not None:
            print("Expected rows:", expected_eval_n)
            if len(eval_df) != expected_eval_n:
                raise ValueError(
                    f"[{dataset}] Expected {expected_eval_n} evaluation rows, found {len(eval_df)}."
                )

        print("\nTarget distribution:")
        print(eval_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_string())

        feature_missing = eval_df[COMBINED_FEATURES].isna().sum().sum()
        target_missing = eval_df[TARGET_COLUMN].isna().sum()
        risk_missing = eval_df[risk_column].isna().sum() if risk_column else None

        print("\nMissing feature values:", feature_missing)
        print("Missing target values:  ", target_missing)
        print("Missing risk values:    ", risk_missing)

        if feature_missing > 0:
            raise ValueError(f"[{dataset}] Reference-free feature matrix contains {feature_missing} missing values.")
        if target_missing > 0:
            raise ValueError(f"[{dataset}] Target contains {target_missing} missing values.")
        if risk_column and risk_missing > 0:
            raise ValueError(f"[{dataset}] Risk column '{risk_column}' contains {risk_missing} missing values.")

        audit(
            dataset, "load_artifact", fit_on="-",
            frozen="existing VoxIntel-R artifact",
            evaluated_on=f"existing evaluation population ({len(eval_df)})",
            note=f"Loaded {resolved_path.name}; risk_column={risk_column}; no artificial split created.",
        )

        return {
            "resolved_path": resolved_path,
            "raw_df": raw_df,
            "val_df": None,
            "test_df": None,
            "eval_df": eval_df,
            "split_col": None,
            "evaluation_mode": evaluation_mode,
            "target_column": TARGET_COLUMN,
            "risk_column": risk_column,
            "available_group_scores": available_group_scores,
        }

    elif evaluation_mode == "split":
        split_col_candidates = [
            c for c in raw_df.columns
            if (c == COLUMN_MAP["split"] or "split" in c.lower() or "subset" in c.lower())
        ]
        if not split_col_candidates:
            raise ValueError(f"[{dataset}] No split column found. Expected a validation/test split identifier.")
        split_col = split_col_candidates[0]

        raw_df["_split_norm"] = raw_df[split_col].astype(str).str.lower().str.strip()
        val_df = raw_df[raw_df["_split_norm"].isin(VALIDATION_SPLIT_VALUES)].reset_index(drop=True)
        test_df = raw_df[raw_df["_split_norm"].isin(TEST_SPLIT_VALUES)].reset_index(drop=True)

        print("\nEvaluation mode:", evaluation_mode)
        print("Split column:", split_col)
        print("Validation rows:", len(val_df))
        print("Test rows:", len(test_df))

        expected_val_n = cfg.get("expected_val_n")
        expected_test_n = cfg.get("expected_test_n")
        if expected_val_n is not None and len(val_df) != expected_val_n:
            raise ValueError(f"[{dataset}] Validation row count mismatch: expected {expected_val_n}, found {len(val_df)}.")
        if expected_test_n is not None and len(test_df) != expected_test_n:
            raise ValueError(f"[{dataset}] Test row count mismatch: expected {expected_test_n}, found {len(test_df)}.")

        print("\nValidation target distribution:")
        print(val_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_string())
        print("\nTest target distribution:")
        print(test_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_string())

        for split_name, split_df in [("validation", val_df), ("test", test_df)]:
            feature_missing = split_df[COMBINED_FEATURES].isna().sum().sum()
            target_missing = split_df[TARGET_COLUMN].isna().sum()
            if feature_missing > 0:
                raise ValueError(f"[{dataset}] {split_name} contains {feature_missing} missing feature values.")
            if target_missing > 0:
                raise ValueError(f"[{dataset}] {split_name} contains {target_missing} missing target values.")
            if risk_column:
                risk_missing = split_df[risk_column].isna().sum()
                if risk_missing > 0:
                    raise ValueError(f"[{dataset}] {split_name} contains {risk_missing} missing risk values.")

        audit(
            dataset, "load_artifact", fit_on=f"validation ({len(val_df)})",
            frozen="frozen VoxIntel-R feature artifact",
            evaluated_on=f"test ({len(test_df)})",
            note=f"Loaded {resolved_path.name}; val={len(val_df)}, test={len(test_df)}; risk_column={risk_column}.",
        )

        return {
            "resolved_path": resolved_path,
            "raw_df": raw_df,
            "val_df": val_df,
            "test_df": test_df,
            "eval_df": None,
            "split_col": split_col,
            "evaluation_mode": evaluation_mode,
            "target_column": TARGET_COLUMN,
            "risk_column": risk_column,
            "available_group_scores": available_group_scores,
        }

    else:
        raise ValueError(f"[{dataset}] Unknown evaluation mode: {evaluation_mode}")


loaded = {}
dataset_load_errors = {}

for ds in DATASETS:
    try:
        loaded[ds] = load_dataset(ds)
    except (FileNotFoundError, ValueError) as e:
        dataset_load_errors[ds] = str(e)
        print()
        print("=" * 80)
        print(f"{ds.upper()} -- LOAD FAILED")
        print("=" * 80)
        print(e)

ACTIVE_DATASETS = [ds for ds in DATASETS if ds in loaded]

print()
print("=" * 80)
print("NOTEBOOK 18 -- DATASET LOAD SUMMARY")
print("=" * 80)
print("\nRequested datasets:", DATASETS)
print("Active datasets:   ", ACTIVE_DATASETS)

if dataset_load_errors:
    print("\nDatasets with errors:")
    for ds, error in dataset_load_errors.items():
        print(f"\n[{ds.upper()}]")
        print(error)

if not ACTIVE_DATASETS:
    raise RuntimeError("No dataset could be loaded. Fix the dataset-specific VoxIntel-R artifacts before continuing.")

print()
print("=" * 80)
print("DATASET CONTRACT SUMMARY")
print("=" * 80)
for ds in ACTIVE_DATASETS:
    info = loaded[ds]
    print(f"\n{ds.upper()}")
    print("  Artifact:", info["resolved_path"])
    print("  Mode:    ", info["evaluation_mode"])
    if info["evaluation_mode"] == "split":
        print("  Validation:", len(info["val_df"]))
        print("  Test:      ", len(info["test_df"]))
    else:
        print("  Evaluation population:", len(info["eval_df"]))
    print("  Target:", info["target_column"])
    print("  Risk column:", info["risk_column"])
    print("  Available group score columns:", info["available_group_scores"] or "none")

print("\nDataset loading complete.")


LOADING SLURP VOXINTEL-R ARTIFACT

Path: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\voxintel_r_predictions.csv
Rows: 8688
Columns: 19

Reference-free feature schema:  PASS
Target column:                   intent_failed
Resolved risk column:            voxintel_r_risk
Forbidden-looking columns present (not used as features): ['ground_truth_intent']
Available group-level score columns (opportunistic, may be empty): {'C': 'voxintel_r_risk'}

Evaluation mode: existing_evaluation_population
Evaluation rows: 8688
Expected rows: 8688

Target distribution:
intent_failed
0    6837
1    1851

Missing feature values: 0
Missing target values:   0
Missing risk values:     0

LOADING FSC VOXINTEL-R ARTIFACT

Path: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_features.csv
Rows: 6911
Columns: 14

Reference-free feature schema:  PASS
Target column:                   intent_failed
Resolved risk column:            None
Forbidden-looking columns present (not used as features): none

## 10b — Artifact audit table

A single machine-readable audit row per dataset, covering exactly what Section 15 of the update brief
asked for: path, shape, columns, target, split availability, available risk/prediction columns, missing
values, feature-schema status, and any forbidden/reference-derived columns spotted in the raw file (even
though they are never used as features). Saved as `notebook18_artifact_audit.csv`.

In [4]:
artifact_audit_rows = []

for ds in ACTIVE_DATASETS:
    info = loaded[ds]
    cfg = DATASET_CONFIG[ds]
    raw_df = info["raw_df"]

    present_forbidden = [
        c for c in raw_df.columns
        if any(tok in c.lower() for tok in FORBIDDEN_FEATURE_SUBSTRINGS)
        and c not in ALLOWED_FEATURE_COLUMNS
    ]

    if info["evaluation_mode"] == "split":
        shape_note = f"validation={len(info['val_df'])}, test={len(info['test_df'])}"
        split_available = True
    else:
        shape_note = f"evaluation_population={len(info['eval_df'])}"
        split_available = False

    artifact_audit_rows.append({
        "dataset": ds,
        "path": str(info["resolved_path"]),
        "raw_rows": len(raw_df),
        "raw_columns": len(raw_df.columns),
        "shape_detail": shape_note,
        "target_column": info["target_column"],
        "split_available": split_available,
        "risk_column": info["risk_column"],
        "available_group_scores": json.dumps(info["available_group_scores"]),
        "missing_feature_values": int(raw_df[COMBINED_FEATURES].isna().sum().sum()),
        "missing_target_values": int(raw_df[info["target_column"]].isna().sum()),
        "feature_schema_pass": all(f in raw_df.columns for f in COMBINED_FEATURES),
        "forbidden_columns_present": json.dumps(present_forbidden),
    })

for ds, err in dataset_load_errors.items():
    artifact_audit_rows.append({
        "dataset": ds,
        "path": str(DATASET_CONFIG[ds]["features_csv"]),
        "raw_rows": None,
        "raw_columns": None,
        "shape_detail": "LOAD FAILED",
        "target_column": TARGET_COLUMN,
        "split_available": None,
        "risk_column": None,
        "available_group_scores": "{}",
        "missing_feature_values": None,
        "missing_target_values": None,
        "feature_schema_pass": False,
        "forbidden_columns_present": "[]",
    })

artifact_audit_df = pd.DataFrame(artifact_audit_rows)
artifact_audit_df.to_csv(OUT_DIR / "notebook18_artifact_audit.csv", index=False)
print(artifact_audit_df.to_string(index=False))
print(f"\nSaved: {OUT_DIR / 'notebook18_artifact_audit.csv'}")

dataset                                                                        path  raw_rows  raw_columns               shape_detail target_column  split_available     risk_column   available_group_scores  missing_feature_values  missing_target_values  feature_schema_pass forbidden_columns_present
  slurp  C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\voxintel_r_predictions.csv      8688           19 evaluation_population=8688 intent_failed            False voxintel_r_risk {"C": "voxintel_r_risk"}                       0                      0                 True   ["ground_truth_intent"]
    fsc C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_features.csv      6911           15 validation=3118, test=3793 intent_failed             True            None                       {}                       0                      0                 True                        []

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\notebook18_artifact_audit.csv


## 11 — Data integrity checks (per dataset)

Runs once per active dataset: row counts vs. frozen expectations, target binariness, feature schema,
forbidden-feature audit, and numeric sanity (no NaN/Inf) -- printed as an explicit PASS/WARN/FAIL table.
For FSC this checks both `validation` and `test`; for SLURP it checks the single evaluation population
and explicitly records that no artificial split was created.

In [5]:
def run_integrity_checks(dataset: str) -> pd.DataFrame:
    cfg = DATASET_CONFIG[dataset]
    info = loaded[dataset]
    raw_df = info["raw_df"]
    target_col = info["target_column"]

    rows = []

    def check(name, passed, detail, hard_fail=False):
        status = "PASS" if passed else ("FAIL" if hard_fail else "WARN")
        rows.append({"dataset": dataset, "check": name, "status": status, "detail": detail})
        if hard_fail and not passed:
            raise AssertionError(f"[{dataset}:{name}] {detail}")

    if cfg["evaluation_mode"] == "split":
        val_df, test_df = info["val_df"], info["test_df"]

        if cfg["expected_val_n"] is not None:
            check("val_row_count", len(val_df) == cfg["expected_val_n"],
                  f"got {len(val_df)}, expected {cfg['expected_val_n']}", hard_fail=True)
        if cfg["expected_test_n"] is not None:
            check("test_row_count", len(test_df) == cfg["expected_test_n"],
                  f"got {len(test_df)}, expected {cfg['expected_test_n']}", hard_fail=True)

        for name, d, expected_fail in [
            ("validation", val_df, cfg["expected_val_failures"]),
            ("test", test_df, cfg["expected_test_failures"]),
        ]:
            vals = set(d[target_col].dropna().unique().tolist())
            check(f"{name}_target_binary", vals.issubset({0, 1}), f"unique values: {vals}", hard_fail=True)
            n_fail = int(d[target_col].sum())
            if expected_fail is not None:
                check(f"{name}_failure_count", n_fail == expected_fail,
                      f"got {n_fail}, expected {expected_fail} (rate {n_fail / len(d):.4%})", hard_fail=True)
            else:
                check(f"{name}_failure_count", True,
                      f"got {n_fail} failures out of {len(d)} (rate {n_fail / len(d):.4%}); no prior expectation")

        evaluation_splits = [("validation", val_df), ("test", test_df)]

        id_col = COLUMN_MAP.get("id")
        if id_col and id_col in raw_df.columns:
            overlap = set(val_df[id_col]) & set(test_df[id_col])
            check("val_test_no_overlap", len(overlap) == 0, f"{len(overlap)} ids appear in both splits", hard_fail=True)
        else:
            check("val_test_no_overlap", True, "no id column configured/found -- skipped (non-blocking)")

    else:
        eval_df = info["eval_df"]
        check("evaluation_row_count", len(eval_df) == cfg["expected_eval_n"],
              f"got {len(eval_df)}, expected {cfg['expected_eval_n']}", hard_fail=True)
        vals = set(eval_df[target_col].dropna().unique().tolist())
        check("evaluation_target_binary", vals.issubset({0, 1}), f"unique values: {vals}", hard_fail=True)
        n_fail = int(eval_df[target_col].sum())
        check("evaluation_failure_count", n_fail >= 0,
              f"got {n_fail} failures out of {len(eval_df)} (rate {n_fail / len(eval_df):.4%})")
        check("artificial_split_check", True,
              "No artificial train/validation/test split created for this existing evaluation population.")
        check("risk_column_present", info["risk_column"] is not None,
              f"resolved risk column: {info['risk_column']}", hard_fail=True)

        evaluation_splits = [("evaluation", eval_df)]

    missing = [c for c in COMBINED_FEATURES if c not in raw_df.columns]
    check("feature_schema", len(missing) == 0, f"missing columns: {missing}", hard_fail=True)

    present_forbidden = [
        c for c in raw_df.columns
        if any(tok in c.lower() for tok in FORBIDDEN_FEATURE_SUBSTRINGS) and c not in ALLOWED_FEATURE_COLUMNS
    ]
    check("forbidden_feature_audit", len(present_forbidden) == 0,
          f"columns matching forbidden tokens (not used as features regardless): {present_forbidden}")

    for name, d in evaluation_splits:
        X = d[COMBINED_FEATURES].apply(pd.to_numeric, errors="coerce")
        n_nan = int(X.isna().sum().sum())
        n_inf = int(np.isinf(X.values).sum())
        check(f"{name}_no_nan", n_nan == 0, f"{n_nan} NaNs found", hard_fail=True)
        check(f"{name}_no_inf", n_inf == 0, f"{n_inf} Infs found", hard_fail=True)

    df = pd.DataFrame(rows)
    n_fail_hard = (df["status"] == "FAIL").sum()
    n_warn = (df["status"] == "WARN").sum()
    audit(dataset, "integrity_checks", fit_on="-", frozen="-", evaluated_on="-",
          note=f"{n_fail_hard} FAIL, {n_warn} WARN, {len(df)} total checks")
    return df


integrity_dfs = {}
for ds in ACTIVE_DATASETS:
    print(f"\n=== [{ds}] integrity checks ===")
    integrity_dfs[ds] = run_integrity_checks(ds)
    print(integrity_dfs[ds].drop(columns=["dataset"]).to_string(index=False))

integrity_df = pd.concat(integrity_dfs.values(), ignore_index=True)
n_fail_hard = (integrity_df["status"] == "FAIL").sum()
n_warn = (integrity_df["status"] == "WARN").sum()
print(f"\nAcross all active datasets: {n_fail_hard} FAIL, {n_warn} WARN out of {len(integrity_df)} checks.")


=== [slurp] integrity checks ===
                   check status                                                                                       detail
    evaluation_row_count   PASS                                                                      got 8688, expected 8688
evaluation_target_binary   PASS                                                                        unique values: {0, 1}
evaluation_failure_count   PASS                                                got 1851 failures out of 8688 (rate 21.3052%)
  artificial_split_check   PASS   No artificial train/validation/test split created for this existing evaluation population.
     risk_column_present   PASS                                                        resolved risk column: voxintel_r_risk
          feature_schema   PASS                                                                          missing columns: []
 forbidden_feature_audit   WARN columns matching forbidden tokens (not used as features reg

## 12 — Shared calibration and selective-prediction utilities

These helper functions are pure evaluation math -- they take arrays of labels and scores and compute a
metric. They do not know or care whether the scores came from a freshly-fit FSC model or an existing
SLURP risk column. They are defined once here and reused by both the SLURP evaluation-only arm
(Sections 13-18) and the FSC validation/frozen-model/test arm (Sections 19-32). None of these functions
themselves constitute "fitting" -- fitting only happens where a function explicitly calls `.fit(...)` on
FSC validation data, later in the notebook.

In [15]:
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score


def expected_calibration_error(y_true, p, n_bins=10):
    '''Equal-width binning ECE, with per-bin diagnostics returned alongside the scalar.'''
    y_true = np.asarray(y_true); p = np.asarray(p)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    n = len(y_true)
    ece = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        cnt = int(mask.sum())
        if cnt == 0:
            rows.append({"bin_lo": lo, "bin_hi": hi, "count": 0, "mean_predicted": np.nan, "observed_freq": np.nan})
            continue
        mean_pred = float(p[mask].mean())
        obs_freq = float(y_true[mask].mean())
        ece += (cnt / n) * abs(obs_freq - mean_pred)
        rows.append({"bin_lo": lo, "bin_hi": hi, "count": cnt, "mean_predicted": mean_pred, "observed_freq": obs_freq})
    return ece, pd.DataFrame(rows)


def max_calibration_error(reliability_df):
    d = reliability_df.dropna(subset=["mean_predicted", "observed_freq"])
    if len(d) == 0:
        return np.nan
    return float((d["mean_predicted"] - d["observed_freq"]).abs().max())


def calibration_report(name, y_true, p):
    brier = brier_score_loss(y_true, p)
    ece, reliability = expected_calibration_error(y_true, p, n_bins=10)
    mce = max_calibration_error(reliability)
    return {"variant": name, "brier": brier, "ece": ece, "mce": mce}, reliability


def risk_coverage_curve(y_true, safety_score, higher_is_safer):
    '''Threshold-free risk-coverage curve: sort by safety, sweep coverage from 0 to 1.
    Needs no fitting -- purely a function of the scores and labels already in hand.'''
    y_true = np.asarray(y_true)
    order = np.argsort(-safety_score if higher_is_safer else safety_score)  # safest first
    y_sorted = y_true[order]
    n = len(y_true)
    coverage = np.arange(1, n + 1) / n
    cum_failures = np.cumsum(y_sorted)
    selective_risk = cum_failures / np.arange(1, n + 1)
    wrong_execution_rate = cum_failures / n
    return pd.DataFrame({
        "coverage": coverage,
        "selective_risk": selective_risk,
        "wrong_execution_rate": wrong_execution_rate,
        "clarification_rate": 1 - coverage,
    })


def area_under_risk_coverage(curve_df, min_coverage=0.0):
    '''Lower AURC = better. Trapezoidal integral of selective_risk over coverage. Threshold-free.'''
    d = curve_df[curve_df["coverage"] >= min_coverage].sort_values("coverage")
    if len(d) < 2:
        return np.nan
    return float(np.trapz(d["selective_risk"], d["coverage"]) / (1 - min_coverage))


def threshold_at_target_coverage(safety_score, higher_is_safer, target_coverage):
    '''Fit-style quantile threshold. Only ever called on a validation split for FSC; for SLURP a
    *descriptive* variant of this same quantile is computed directly on the evaluation population and
    explicitly labeled as such -- never presented as a held-out estimate.'''
    q = (1 - target_coverage) if higher_is_safer else target_coverage
    return float(np.quantile(safety_score, q))


def apply_frozen_threshold(y_true, score, higher_is_safer, tau):
    execute = (score >= tau) if higher_is_safer else (score <= tau)
    coverage = execute.mean()
    selective_risk = y_true[execute].mean() if execute.sum() > 0 else np.nan
    wrong_execution_rate = ((execute) & (y_true == 1)).sum() / len(y_true)
    clarification_rate = 1 - coverage
    return {"coverage": coverage, "selective_risk": selective_risk,
            "wrong_execution_rate": wrong_execution_rate, "clarification_rate": clarification_rate}


N_BOOTSTRAP = 2000


def paired_bootstrap_delta(y_true, p_a, p_b, metric_fn, rng, n_boot=N_BOOTSTRAP):
    '''Resamples ROWS jointly across two already-computed score arrays. This needs no fitting -- it is a
    valid evaluation-only operation for SLURP as well as FSC.'''
    y_true = np.asarray(y_true); p_a = np.asarray(p_a); p_b = np.asarray(p_b)
    deltas = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y_s = y_true[idx]
        if y_s.sum() == 0 or y_s.sum() == n:
            continue
        try:
            deltas.append(metric_fn(y_s, p_b[idx]) - metric_fn(y_s, p_a[idx]))
        except ValueError:
            continue
    deltas = np.array(deltas)
    if len(deltas) == 0:
        return np.nan, np.nan, np.nan
    return float(deltas.mean()), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))


def paired_bootstrap_aurc_delta(y_true, score_risk, higher_is_safer_risk,
                                 score_baseline, higher_is_safer_baseline,
                                 rng, n_boot=N_BOOTSTRAP, min_coverage=0.0):
    '''DeltaAURC = AURC(risk) - AURC(baseline). AURC: lower = better, so a negative delta means the risk
    score gives a better (lower-risk) risk-coverage trade-off than the baseline in that resample.
    `score_baseline=None` represents the always-execute baseline, whose AURC is just the resampled
    failure rate (no score/threshold involved). Used for both SLURP (Section 15) and FSC (Section 27b) --
    this is a shared, dataset-agnostic evaluation-only operation: it resamples already-computed scores
    and labels and fits nothing, so it is valid on SLURP's existing-evaluation-population artifact as
    well as on FSC's held-out test split.'''
    y_true = np.asarray(y_true); score_risk = np.asarray(score_risk)
    if score_baseline is not None:
        score_baseline = np.asarray(score_baseline)
    n = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y_s = y_true[idx]
        if y_s.sum() == 0 or y_s.sum() == n:
            continue
        curve_risk = risk_coverage_curve(y_s, score_risk[idx], higher_is_safer_risk)
        aurc_risk = area_under_risk_coverage(curve_risk, min_coverage=min_coverage)
        if score_baseline is None:
            aurc_baseline = float(y_s.mean())
        else:
            curve_baseline = risk_coverage_curve(y_s, score_baseline[idx], higher_is_safer_baseline)
            aurc_baseline = area_under_risk_coverage(curve_baseline, min_coverage=min_coverage)
        if np.isnan(aurc_risk) or np.isnan(aurc_baseline):
            continue
        deltas.append(aurc_risk - aurc_baseline)
    deltas = np.array(deltas)
    if len(deltas) == 0:
        return np.nan, np.nan, np.nan
    return float(deltas.mean()), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))


def run_aurc_bootstrap_suite(y_true, risk_scores, higher_is_safer_risk, risk_name, baseline_signals, rng):
    '''baseline_signals: dict of {name: {"scores": array_or_None, "higher_is_safer": bool}}, where
    scores=None means the always-execute baseline. Returns a tidy DataFrame with one row per comparison,
    a "significant" flag (95% CI on delta excludes zero), and a "favors_risk" flag (significant AND the
    risk score's AURC is the lower one). Shared by SLURP (Section 15) and FSC (Section 27b) so both
    datasets' H3 significance tests use the same code path and the same metric (AURC, matching the
    quantity H3 is actually a claim about -- see Section 07).'''
    rows = []
    for name, cfg in baseline_signals.items():
        mean_d, lo, hi = paired_bootstrap_aurc_delta(
            y_true, risk_scores, higher_is_safer_risk, cfg["scores"], cfg["higher_is_safer"], rng)
        significant = not (lo <= 0 <= hi)
        favors_risk = bool(significant and mean_d < 0)
        rows.append({"comparison": f"{risk_name} vs {name}", "metric": "aurc",
                      "delta_mean": mean_d, "ci_lo": lo, "ci_hi": hi,
                      "significant": significant, "favors_risk": favors_risk})
    return pd.DataFrame(rows)


def verdict_h3(aurc_series, bootstrap_df, risk_signal, baseline_signals):
    '''Shared H3 decision rule for both datasets (Section 33). AURC point estimates decide whether the
    risk score "beats" each baseline; the AURC bootstrap decides whether any of those wins are
    statistically distinguishable from zero. A win on every point estimate that is NOT statistically
    distinguishable from zero on any comparison is reported as INCONCLUSIVE rather than SUPPORTED -- a
    ~0.00002 AURC gap on 23 FSC test positives is not, by itself, evidence.'''
    beats = {b: bool(aurc_series.loc[risk_signal] < aurc_series.loc[b]) for b in baseline_signals}
    beats_all = all(beats.values())
    beats_always = beats.get("always_execute", False)

    bt = bootstrap_df.set_index("comparison") if bootstrap_df is not None and len(bootstrap_df) else None
    any_significant_favor = False
    if bt is not None:
        for b in baseline_signals:
            key = f"{risk_signal} vs {b}"
            if key in bt.index and bool(bt.loc[key, "favors_risk"]):
                any_significant_favor = True

    if beats_all and any_significant_favor:
        return "SUPPORTED"
    elif beats_all and not any_significant_favor:
        return "INCONCLUSIVE"
    elif beats_always:
        return "PARTIALLY SUPPORTED"
    return "NOT SUPPORTED"


print("Shared calibration / selective-prediction utilities defined.")
print("These are pure functions of (labels, scores) -- no dataset-specific state.")

Shared calibration / selective-prediction utilities defined.
These are pure functions of (labels, scores) -- no dataset-specific state.


## 13 — SLURP evaluation, part 1: H2 (calibration of the *existing* VoxIntel-R risk)

**SLURP evaluation-only.** Nothing is fit here. `R["slurp"]` is initialized and will only ever contain
fields that are actually valid for an existing-evaluation-population dataset -- no `y_val`, no
`val_scores`, no fitted calibrator.

This reports Brier score, ECE, MCE, and a reliability table for SLURP's existing risk column, evaluated
against the full 8,688-row population. This reproduces the kind of result the project already established
for SLURP (existing risk: Brier ≈0.1025, ECE ≈0.0192, MCE ≈0.0841) -- but as a *live* computation against
whatever is actually in `voxintel_r_predictions.csv` right now, not a hard-coded number.

In [16]:
R = {}  # top-level per-dataset results accumulator; fields differ by evaluation_mode -- see Section 08

if "slurp" in ACTIVE_DATASETS:
    slurp_info = loaded["slurp"]
    eval_df = slurp_info["eval_df"]
    risk_col = slurp_info["risk_column"]
    target_col = slurp_info["target_column"]

    y_eval = eval_df[target_col].astype(int).values
    existing_risk = eval_df[risk_col].astype(float).values

    raw_report, raw_reliability = calibration_report("existing_voxintel_r", y_eval, existing_risk)

    R["slurp"] = {
        "evaluation_mode": slurp_info["evaluation_mode"],
        "target_col": target_col,
        "risk_column": risk_col,
        "y_eval": y_eval,
        "existing_risk": existing_risk,
        "n_eval": len(eval_df),
        "n_failures": int(y_eval.sum()),
        "failure_rate": float(y_eval.mean()),
        "h2_report": raw_report,
        "h2_reliability": raw_reliability,
    }

    audit("slurp", "calibration_evaluation", fit_on="-",
          frozen="existing VoxIntel-R risk (no new calibrator fit)",
          evaluated_on=f"existing evaluation population ({len(eval_df)})",
          note="H2 evaluated on existing risk only; SLURP has no validation split to fit sigmoid/isotonic on.")

    print("=== [slurp] H2 -- calibration of existing VoxIntel-R risk ===")
    print(f"Population: {len(eval_df)}, failures: {int(y_eval.sum())}, rate: {y_eval.mean():.4%}")
    print(pd.DataFrame([raw_report]).to_string(index=False))
    print("\nReliability table (existing risk):")
    print(raw_reliability.to_string(index=False))
else:
    print("SLURP not active this run -- see Section 10 load errors. Skipping SLURP H2.")

=== [slurp] H2 -- calibration of existing VoxIntel-R risk ===
Population: 8688, failures: 1851, rate: 21.3052%
            variant  brier      ece      mce
existing_voxintel_r 0.1025 0.019151 0.084089

Reliability table (existing risk):
 bin_lo  bin_hi  count  mean_predicted  observed_freq
    0.0     0.1   4336        0.042322       0.033672
    0.1     0.2   1515        0.140224       0.124752
    0.2     0.3    697        0.243496       0.263989
    0.3     0.4    428        0.350491       0.434579
    0.4     0.5    434        0.446129       0.490783
    0.5     0.6    354        0.549134       0.559322
    0.6     0.7    315        0.648487       0.701587
    0.7     0.8    275        0.748182       0.774545
    0.8     0.9    219        0.844673       0.872146
    0.9     1.0    115        0.937536       0.956522


## 14 — SLURP evaluation, part 2: H3 (selective prediction)

The risk-coverage curve and AURC are **threshold-free** -- they only require sorting the existing risk
score and sweeping coverage, which needs no fitting and is therefore legitimate to compute directly on
SLURP's evaluation population. ROC-AUC and PR-AUC (rank-quality metrics) are reported the same way.

A **single frozen operating point at 90% coverage** (as used for FSC in Section 26) is **not** reported
here in the same way, because selecting *that* threshold and then reporting its performance both require
the same labels -- there is no independent split to select it on for SLURP. Instead, a *descriptive*
operating point is shown: "if you executed the safest 90% of this population by existing risk, what would
selective risk look like *in this population*" -- explicitly labeled as descriptive, not a held-out
generalization estimate.

In [18]:
if "slurp" in ACTIVE_DATASETS:
    eval_df = slurp_info["eval_df"]
    y_eval = R["slurp"]["y_eval"]
    existing_risk = R["slurp"]["existing_risk"]

    SLURP_SIGNALS = {
        "always_execute": None,
        "asr_confidence": {"scores": eval_df["asr_mean_confidence"].values, "higher_is_safer": True},
        "intent_confidence": {"scores": eval_df["intent_confidence"].values, "higher_is_safer": True},
        "voxintel_r_existing_risk": {"scores": existing_risk, "higher_is_safer": False},
    }

    curves, aurc_rows, rank_metric_rows, descriptive_points = {}, [], [], []

    for signal, cfg in SLURP_SIGNALS.items():
        if signal == "always_execute":
            curve = pd.DataFrame({"coverage": [1.0], "selective_risk": [y_eval.mean()],
                                   "wrong_execution_rate": [y_eval.mean()], "clarification_rate": [0.0]})
            aurc = float(y_eval.mean())
            roc_auc = pr_auc = np.nan
        else:
            curve = risk_coverage_curve(y_eval, cfg["scores"], cfg["higher_is_safer"])
            aurc = area_under_risk_coverage(curve)
            score_for_auc = cfg["scores"] if cfg["higher_is_safer"] else -cfg["scores"]
            roc_auc = roc_auc_score(y_eval, score_for_auc)
            pr_auc = average_precision_score(y_eval, score_for_auc)

            tau_desc = threshold_at_target_coverage(cfg["scores"], cfg["higher_is_safer"], TARGET_COVERAGE)
            op = apply_frozen_threshold(y_eval, cfg["scores"], cfg["higher_is_safer"], tau_desc)
            op["signal"] = signal
            op["tau_descriptive"] = tau_desc
            descriptive_points.append(op)

        curves[signal] = curve
        aurc_rows.append({"signal": signal, "aurc": aurc})
        rank_metric_rows.append({"signal": signal, "roc_auc": roc_auc, "pr_auc": pr_auc})

    aurc_df = pd.DataFrame(aurc_rows)
    rank_metrics_df = pd.DataFrame(rank_metric_rows)
    descriptive_points_df = pd.DataFrame(descriptive_points)[
        ["signal", "tau_descriptive", "coverage", "selective_risk", "wrong_execution_rate", "clarification_rate"]
    ] if descriptive_points else pd.DataFrame()

    R["slurp"].update(curves=curves, aurc_df=aurc_df, rank_metrics_df=rank_metrics_df,
                       descriptive_points_df=descriptive_points_df)

    audit("slurp", "selective_prediction_evaluation", fit_on="-",
          frozen="threshold-free curves on existing risk; descriptive-only operating point",
          evaluated_on=f"existing evaluation population ({len(eval_df)})",
          note="No threshold fit on a held-out split -- SLURP has none. Descriptive operating point is "
               "computed and evaluated on the same population and is labeled accordingly, not treated as "
               "a generalization estimate.")

    print("=== [slurp] H3 -- AURC by signal (threshold-free, lower=better) ===")
    print(aurc_df.to_string(index=False))
    print("\n=== [slurp] H3 -- rank-quality metrics by signal ===")
    print(rank_metrics_df.to_string(index=False))
    print(f"\n=== [slurp] H3 -- DESCRIPTIVE operating point at target coverage={TARGET_COVERAGE} ===")
    print("(computed and evaluated on the SAME population -- not a held-out estimate)")
    print(descriptive_points_df.to_string(index=False))

    aurc_df.assign(dataset="slurp").to_csv(OUT_DIR / "slurp_voxintel_r_aurc.csv", index=False)
    rank_metrics_df.assign(dataset="slurp").to_csv(OUT_DIR / "slurp_voxintel_r_rank_metrics.csv", index=False)
    descriptive_points_df.assign(dataset="slurp").to_csv(
        OUT_DIR / "slurp_voxintel_r_descriptive_operating_points.csv", index=False)
    print(f"\nSaved: {OUT_DIR / 'slurp_voxintel_r_aurc.csv'}, "
          f"{OUT_DIR / 'slurp_voxintel_r_rank_metrics.csv'}, "
          f"{OUT_DIR / 'slurp_voxintel_r_descriptive_operating_points.csv'}")
else:
    print("SLURP not active this run. Skipping SLURP H3.")

=== [slurp] H3 -- AURC by signal (threshold-free, lower=better) ===
                  signal     aurc
          always_execute 0.213052
          asr_confidence 0.187328
       intent_confidence 0.083179
voxintel_r_existing_risk 0.058600

=== [slurp] H3 -- rank-quality metrics by signal ===
                  signal  roc_auc   pr_auc
          always_execute      NaN      NaN
          asr_confidence 0.406565 0.190859
       intent_confidence 0.180746 0.127550
voxintel_r_existing_risk 0.117385 0.121304

=== [slurp] H3 -- DESCRIPTIVE operating point at target coverage=0.9 ===
(computed and evaluated on the SAME population -- not a held-out estimate)
                  signal  tau_descriptive  coverage  selective_risk  wrong_execution_rate  clarification_rate
          asr_confidence         0.939730  0.899977        0.191585              0.172422            0.100023
       intent_confidence         0.615416  0.899977        0.152833              0.137546            0.100023
voxintel_r_exi

## 15 — SLURP evaluation, part 3: statistical uncertainty for H3

H3 is a claim about **selective prediction / risk-coverage performance** (Section 07), and Section 14's
point-estimate comparison uses **AURC** (area under the risk-coverage curve) as the primary quantity for
that reason. The significance test here therefore bootstraps **ΔAURC** directly (`paired_bootstrap_aurc_delta`,
Section 12), rather than a different metric (ROC-AUC) that answers a related-but-not-identical question --
ROC-AUC does not distinguish where along the coverage axis two signals disagree, whereas AURC weighs the
whole risk-coverage trade-off. Paired bootstrap over rows (2,000 resamples): for each resample, the entire
risk-coverage curve and its AURC are recomputed for the existing VoxIntel-R risk and for each baseline
(including always-execute), and the paired difference is recorded. This needs no fitting -- it resamples
already-computed scores and labels -- so it is valid on SLURP's evaluation population even without a
validation split. The same shared bootstrap function is used for FSC in Section 27b.

In [19]:
if "slurp" in ACTIVE_DATASETS:
    eval_df = slurp_info["eval_df"]
    y_eval = R["slurp"]["y_eval"]
    existing_risk = R["slurp"]["existing_risk"]
    rng = np.random.default_rng(RANDOM_SEED)

    slurp_baseline_signals = {
        "always_execute": {"scores": None, "higher_is_safer": True},
        "asr_confidence": {"scores": eval_df["asr_mean_confidence"].values, "higher_is_safer": True},
        "intent_confidence": {"scores": eval_df["intent_confidence"].values, "higher_is_safer": True},
    }
    slurp_bootstrap_df = run_aurc_bootstrap_suite(
        y_eval, existing_risk, False, "voxintel_r_existing_risk", slurp_baseline_signals, rng)
    R["slurp"]["bootstrap_df"] = slurp_bootstrap_df
    slurp_bootstrap_df.to_csv(OUT_DIR / "slurp_voxintel_r_bootstrap_h3.csv", index=False)

    print("=== [slurp] paired bootstrap AURC deltas (existing risk vs baselines; negative delta = risk is better) ===")
    print(slurp_bootstrap_df.to_string(index=False))
    print(f"\nSaved: {OUT_DIR / 'slurp_voxintel_r_bootstrap_h3.csv'}")
else:
    print("SLURP not active this run. Skipping SLURP bootstrap.")

=== [slurp] paired bootstrap AURC deltas (existing risk vs baselines; negative delta = risk is better) ===
                                   comparison metric  delta_mean     ci_lo     ci_hi  significant  favors_risk
   voxintel_r_existing_risk vs always_execute   aurc   -0.154534 -0.160891 -0.148075         True         True
   voxintel_r_existing_risk vs asr_confidence   aurc   -0.128947 -0.139604 -0.118522         True         True
voxintel_r_existing_risk vs intent_confidence   aurc   -0.024600 -0.029119 -0.020349         True         True

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_bootstrap_h3.csv


## 16 — SLURP evaluation, part 4: H4 (cost-sensitive decision making) -- **not evaluable**

H4 asks whether a severity/cost-*optimized* threshold beats a uniform threshold. Optimizing a threshold
means picking the value that minimizes expected cost **against labels** -- which is a fitting step. Doing
that and then reporting the resulting cost on the same population it was optimized against is exactly the
kind of leakage this notebook is required to avoid (Section 08), and SLURP has no independent split to
optimize on and then evaluate elsewhere.

This is stated explicitly here, and carried through as `INCONCLUSIVE` in Section 33/34, rather than
computing a cost number that would look like a fair comparison but is not one.

In [20]:
if "slurp" in ACTIVE_DATASETS:
    R["slurp"]["h4_verdict"] = "INCONCLUSIVE"
    R["slurp"]["h4_detail"] = (
        "Not evaluated: cost-optimal threshold selection requires optimizing against labels, which needs "
        "an independent split to avoid evaluating the threshold on the same labels used to pick it. "
        "SLURP's artifact (existing evaluation population) provides no such split."
    )
    audit("slurp", "cost_sensitive_evaluation", fit_on="-", frozen="-", evaluated_on="-",
          note=R["slurp"]["h4_detail"])
    print("[slurp] H4 verdict: INCONCLUSIVE")
    print(R["slurp"]["h4_detail"])
else:
    print("SLURP not active this run. Skipping SLURP H4.")

[slurp] H4 verdict: INCONCLUSIVE
Not evaluated: cost-optimal threshold selection requires optimizing against labels, which needs an independent split to avoid evaluating the threshold on the same labels used to pick it. SLURP's artifact (existing evaluation population) provides no such split.


## 17 — SLURP evaluation, part 5: subgroup analysis and opportunistic group-level comparison

Two optional analyses, each run only if the underlying column actually exists in the artifact --
otherwise explicitly reported as unavailable rather than proxied:

1. **Intent-level subgroup failure rates**, if an intent-identity column can be found (ground-truth intent
   used here only for grouping/description, never as a predictive feature -- consistent with Section 02).
2. **Per-feature-group (A/B/C) risk comparison**, only if SLURP's artifact happens to already contain
   separate prediction columns for those groups (`GROUP_SCORE_COLUMN_CANDIDATES`, checked in Section 10).
   The old version of this notebook assumed `R["slurp"]["test_scores"]["A"/"B"/"C"]` always existed; this
   version checks first and reports "unavailable" if it does not, rather than fabricating a comparison.

In [21]:
if "slurp" in ACTIVE_DATASETS:
    eval_df = slurp_info["eval_df"]
    y_eval = R["slurp"]["y_eval"]
    target_col = R["slurp"]["target_col"]

    # --- 1. Intent-level subgroup failure rates (descriptive only) ---
    intent_col = COLUMN_MAP.get("intent_label")
    if not intent_col:
        auto = [c for c in eval_df.columns
                if "intent" in c.lower() and c not in ALLOWED_FEATURE_COLUMNS and c != target_col]
        intent_col = auto[0] if auto else None

    if intent_col and intent_col in eval_df.columns:
        subgroup_df = (
            eval_df.groupby(intent_col)[target_col]
            .agg(n="count", failures="sum")
            .assign(failure_rate=lambda d: d["failures"] / d["n"])
            .sort_values("failure_rate", ascending=False)
            .reset_index()
        )
        R["slurp"]["subgroup_df"] = subgroup_df
        subgroup_df.to_csv(OUT_DIR / "slurp_voxintel_r_intent_subgroup_failure_rates.csv", index=False)
        print(f"=== [slurp] intent-level failure rate (grouped by '{intent_col}', descriptive only) ===")
        print(subgroup_df.head(15).to_string(index=False))
        print(f"\nSaved: {OUT_DIR / 'slurp_voxintel_r_intent_subgroup_failure_rates.csv'}")
    else:
        R["slurp"]["subgroup_df"] = None
        print("[slurp] No intent-identity column found in the artifact -- subgroup analysis UNAVAILABLE "
              "(not proxied).")

    # --- 2. Opportunistic per-group (A/B/C) comparison ---
    available_group_scores = slurp_info["available_group_scores"]
    if len(available_group_scores) >= 2:
        group_rows = []
        for group, col in available_group_scores.items():
            scores = eval_df[col].astype(float).values
            group_rows.append({
                "group": group, "column": col,
                "roc_auc": roc_auc_score(y_eval, scores),
                "pr_auc": average_precision_score(y_eval, scores),
                "brier": brier_score_loss(y_eval, scores),
            })
        slurp_group_comparison_df = pd.DataFrame(group_rows)
        R["slurp"]["group_comparison_df"] = slurp_group_comparison_df
        slurp_group_comparison_df.to_csv(OUT_DIR / "slurp_voxintel_r_group_comparison.csv", index=False)
        print(f"\n=== [slurp] opportunistic per-group comparison (columns found: {available_group_scores}) ===")
        print(slurp_group_comparison_df.to_string(index=False))
        print(f"\nSaved: {OUT_DIR / 'slurp_voxintel_r_group_comparison.csv'}")
    else:
        R["slurp"]["group_comparison_df"] = None
        print(f"\n[slurp] Fewer than 2 per-group prediction columns found "
              f"(found: {available_group_scores}) -- per-group A/B/C comparison UNAVAILABLE on SLURP "
              f"(not proxied; only the single existing combined risk column is evaluated above).")
else:
    print("SLURP not active this run. Skipping SLURP subgroup analysis.")

=== [slurp] intent-level failure rate (grouped by 'ground_truth_intent', descriptive only) ===
ground_truth_intent  n  failures  failure_rate
             quirky 26        26      1.000000
           cleaning  8         8      1.000000
              greet  2         2      1.000000
      general_greet 17        17      1.000000
            factoid 14        14      1.000000
                set  3         3      1.000000
        hue_lightup  1         1      1.000000
              query 87        87      1.000000
      cooking_query 14        14      1.000000
       hue_lightoff  9         9      1.000000
              music 39        39      1.000000
  music_dislikeness  5         5      1.000000
           podcasts  6         6      1.000000
               post  2         2      1.000000
    iot_hue_lighton 11        10      0.909091

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_intent_subgroup_failure_rates.csv

[slurp] Fewer than 2 per-group prediction col

## 18 — SLURP evaluation summary

Consolidates everything computed for SLURP (Sections 13-17) into a single row/table and saves
`slurp_voxintel_r_evaluation_summary.csv` and `notebook18_slurp_evaluation.csv`. This closes the SLURP arm
of the notebook -- Sections 19 onward switch entirely to FSC's validation -> frozen model -> test protocol.

In [22]:
if "slurp" in ACTIVE_DATASETS:
    s = R["slurp"]
    slurp_summary_row = {
        "dataset": "slurp",
        "evaluation_mode": s["evaluation_mode"],
        "n_eval": s["n_eval"],
        "n_failures": s["n_failures"],
        "failure_rate": s["failure_rate"],
        "risk_column": s["risk_column"],
        "h2_brier": s["h2_report"]["brier"],
        "h2_ece": s["h2_report"]["ece"],
        "h2_mce": s["h2_report"]["mce"],
        "h3_aurc_existing_risk": s["aurc_df"].set_index("signal").loc["voxintel_r_existing_risk", "aurc"],
        "h3_aurc_always_execute": s["aurc_df"].set_index("signal").loc["always_execute", "aurc"],
        "h3_aurc_asr_confidence": s["aurc_df"].set_index("signal").loc["asr_confidence", "aurc"],
        "h3_aurc_intent_confidence": s["aurc_df"].set_index("signal").loc["intent_confidence", "aurc"],
        "h4_verdict": s["h4_verdict"],
        "subgroup_analysis_available": s["subgroup_df"] is not None,
        "group_comparison_available": s["group_comparison_df"] is not None,
    }
    slurp_summary_df = pd.DataFrame([slurp_summary_row])
    R["slurp"]["summary_df"] = slurp_summary_df

    slurp_summary_df.to_csv(OUT_DIR / "slurp_voxintel_r_evaluation_summary.csv", index=False)
    slurp_summary_df.to_csv(OUT_DIR / "notebook18_slurp_evaluation.csv", index=False)

    print("=== [slurp] evaluation summary ===")
    print(slurp_summary_df.T.to_string(header=False))
    print(f"\nSaved: {OUT_DIR / 'slurp_voxintel_r_evaluation_summary.csv'}")
    print(f"Saved: {OUT_DIR / 'notebook18_slurp_evaluation.csv'}")
else:
    print("SLURP not active this run -- no summary produced. See Section 10 for the load error.")

print("\n" + "=" * 80)
print("SLURP ARM COMPLETE. Switching to FSC's validation -> frozen model -> test protocol.")
print("=" * 80)

=== [slurp] evaluation summary ===
dataset                                               slurp
evaluation_mode              existing_evaluation_population
n_eval                                                 8688
n_failures                                             1851
failure_rate                                       0.213052
risk_column                                 voxintel_r_risk
h2_brier                                             0.1025
h2_ece                                             0.019151
h2_mce                                             0.084089
h3_aurc_existing_risk                                0.0586
h3_aurc_always_execute                             0.213052
h3_aurc_asr_confidence                             0.187328
h3_aurc_intent_confidence                          0.083179
h4_verdict                                     INCONCLUSIVE
subgroup_analysis_available                            True
group_comparison_available                            False

Save

## 19 — FSC risk modeling: fit A/B/C on validation, freeze, evaluate once on test

FSC has a genuine validation/test split, so this is where the "fit on validation, evaluate once on test"
protocol actually applies. For each feature group (A = ASR-only, B = intent-only, C = combined):

1. **5-fold stratified CV on validation, producing out-of-fold (OOF) probabilities** for every validation
   row -- each row's probability comes from a fold-model that never saw that row's label during training.
   These OOF probabilities, not the in-sample predictions of a model fit on all of validation, are what
   Sections 21-23 fit and select calibration on, and what Section 26 selects a selective-prediction
   threshold on. Fitting a calibrator directly on a model's in-sample predictions on the same rows it was
   trained on is optimistic -- the model has already partly memorized those rows -- so this notebook does
   not do that.
2. Refit on **all of validation** -> frozen model (used only to score test, once).
3. **Single** `predict_proba` pass on **test**, from the frozen model.

Random Forest is reused as the model family (selected in Notebook 17; see Section 04) -- not re-derived
here, only refit under this notebook's own validation/test discipline.

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

FEATURE_GROUPS = {"A": ASR_FEATURES, "B": INTENT_FEATURES, "C": COMBINED_FEATURES}

def make_rf(seed=RANDOM_SEED):
    return RandomForestClassifier(n_estimators=300, random_state=seed, class_weight="balanced", n_jobs=-1)

def fit_fsc_risk_models(val_df, test_df, target_col: str) -> dict:
    y_val = val_df[target_col].astype(int).values
    y_test = test_df[target_col].astype(int).values

    cv_summary = []
    frozen_models, oof_val_scores, test_scores = {}, {}, {}
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    for name, feats in FEATURE_GROUPS.items():
        X_val = val_df[feats].values
        X_test = test_df[feats].values

        # Out-of-fold validation predictions: each fold's held-out rows are scored by a
        # model trained only on the OTHER folds. This is the array everything downstream
        # (calibration fitting/selection, threshold selection) is required to use instead
        # of a full-validation model's in-sample predict_proba.
        oof = np.full(len(y_val), np.nan, dtype=float)
        fold_aucs = []
        for train_idx, holdout_idx in skf.split(X_val, y_val):
            fold_model = make_rf()
            fold_model.fit(X_val[train_idx], y_val[train_idx])
            fold_proba = fold_model.predict_proba(X_val[holdout_idx])[:, 1]
            oof[holdout_idx] = fold_proba
            fold_aucs.append(roc_auc_score(y_val[holdout_idx], fold_proba))
        assert not np.isnan(oof).any(), f"[{name}] OOF predictions incomplete -- every validation row must be held out exactly once."
        fold_aucs = np.array(fold_aucs)
        oof_val_scores[name] = oof
        cv_summary.append({"model": name, "n_features": len(feats),
                            "cv_roc_auc_mean": fold_aucs.mean(), "cv_roc_auc_std": fold_aucs.std()})

        model = make_rf()
        model.fit(X_val, y_val)  # frozen: refit on ALL validation data, never touches test
        frozen_models[name] = model
        test_scores[name] = model.predict_proba(X_test)[:, 1]  # single evaluation on test

        audit("fsc", f"risk_model_{name}",
              fit_on="validation (5-fold CV -> out-of-fold probabilities; full-val refit for freeze)",
              frozen=f"RandomForestClassifier over {feats}", evaluated_on="test (single pass)",
              note="Calibration/threshold steps downstream use the OOF probabilities, not the frozen "
                   "model's in-sample validation predictions.")

    cv_summary_df = pd.DataFrame(cv_summary)
    return {"y_val": y_val, "y_test": y_test, "cv_summary_df": cv_summary_df,
            "frozen_models": frozen_models, "oof_val_scores": oof_val_scores, "test_scores": test_scores}

if "fsc" in ACTIVE_DATASETS:
    fsc_info = loaded["fsc"]
    target_col = fsc_info["target_column"]
    fit_out = fit_fsc_risk_models(fsc_info["val_df"], fsc_info["test_df"], target_col)
    R["fsc"] = {"evaluation_mode": fsc_info["evaluation_mode"], "target_col": target_col, **fit_out}

    R["fsc"]["cv_summary_df"].to_csv(OUT_DIR / "fsc_voxintel_r_cv_results.csv", index=False)
    R["fsc"]["cv_summary_df"].to_csv(OUT_DIR / "notebook18_fsc_cv_results.csv", index=False)
    print("=== [fsc] 5-fold CV ROC-AUC by feature group (validation, out-of-fold) ===")
    print(fit_out["cv_summary_df"].to_string(index=False))
    print(f"\nSaved: {OUT_DIR / 'fsc_voxintel_r_cv_results.csv'}")
else:
    print("FSC not active this run -- see Section 10 load errors. Skipping all FSC sections.")

=== [fsc] 5-fold CV ROC-AUC by feature group (validation, out-of-fold) ===
model  n_features  cv_roc_auc_mean  cv_roc_auc_std
    A           9         0.824270        0.035460
    B           3         0.753656        0.045914
    C          12         0.844243        0.045544

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_cv_results.csv


## 20 — FSC raw (uncalibrated) risk scores

Before applying any post-hoc calibration, this reports the calibration quality of the raw Random Forest
`predict_proba` output for model C, on FSC's own frozen test split. This is the baseline Sections 21-22
try to improve on.

In [24]:
if "fsc" in ACTIVE_DATASETS:
    y_test = R["fsc"]["y_test"]; test_scores = R["fsc"]["test_scores"]
    raw_report, raw_reliability = calibration_report("raw_C", y_test, test_scores["C"])
    R["fsc"]["raw_report"] = raw_report
    R["fsc"]["raw_reliability"] = raw_reliability
    print("=== [fsc] raw model-C calibration (test) ===")
    print(pd.DataFrame([raw_report]).to_string(index=False))
    print(f"Note: FSC test split has {int(y_test.sum())} positive (failure) cases out of {len(y_test)} -- "
          f"high-risk bins may be sparse or empty. See Section 32 for uncertainty.")

=== [fsc] raw model-C calibration (test) ===
variant    brier      ece      mce
  raw_C 0.005585 0.008162 0.263333
Note: FSC test split has 23 positive (failure) cases out of 3793 -- high-risk bins may be sparse or empty. See Section 32 for uncertainty.


## 21 — FSC post-hoc sigmoid (Platt) calibration -- the pre-specified primary method

**Sigmoid calibration is pre-specified as model C's primary calibration method for FSC** (`FSC_PRIMARY_CALIBRATION_VARIANT`,
Section 09), decided before looking at any ECE/Brier/MCE number, rather than picked by whichever variant
happens to score lowest on validation. Section 22's isotonic calibration is fit and reported everywhere
this is, but only as a *sensitivity analysis* -- see Section 23 for why a data-driven ECE-based pick
between raw/sigmoid/isotonic is not used here.

A logistic regression is fit from `logit(out-of-fold validation risk score)` -> `validation label`, fit on
**FSC validation's out-of-fold (OOF) predictions from Section 19** (never the in-sample predictions of a
model fit on all of validation), then applied frozen to FSC test scores.

In [25]:
if "fsc" in ACTIVE_DATASETS:
    from sklearn.linear_model import LogisticRegression

    def _to_logit(p, eps=1e-6):
        p = np.clip(p, eps, 1 - eps)
        return np.log(p / (1 - p))

    def fit_sigmoid_calibrator(p_val, y_val):
        lr = LogisticRegression()
        lr.fit(_to_logit(p_val).reshape(-1, 1), y_val)
        return lr

    def apply_sigmoid_calibrator(calibrator, p):
        return calibrator.predict_proba(_to_logit(p).reshape(-1, 1))[:, 1]

    y_val = R["fsc"]["y_val"]; oof_val_scores = R["fsc"]["oof_val_scores"]; test_scores = R["fsc"]["test_scores"]
    sigmoid_calibrators, sigmoid_oof_val_scores, sigmoid_test_scores = {}, {}, {}
    for name in FEATURE_GROUPS:
        cal = fit_sigmoid_calibrator(oof_val_scores[name], y_val)
        sigmoid_calibrators[name] = cal
        sigmoid_oof_val_scores[name] = apply_sigmoid_calibrator(cal, oof_val_scores[name])
        sigmoid_test_scores[name] = apply_sigmoid_calibrator(cal, test_scores[name])
        audit("fsc", f"sigmoid_calibration_{name}", fit_on="validation (out-of-fold predictions)",
              frozen="LogisticRegression on logit(OOF risk)", evaluated_on="test (single pass)")

    R["fsc"]["sigmoid_calibrators"] = sigmoid_calibrators
    R["fsc"]["sigmoid_oof_val_scores"] = sigmoid_oof_val_scores
    R["fsc"]["sigmoid_test_scores"] = sigmoid_test_scores
    print("[fsc] Sigmoid calibration fit (on out-of-fold validation predictions) for models:",
          list(sigmoid_calibrators.keys()))

[fsc] Sigmoid calibration fit (on out-of-fold validation predictions) for models: ['A', 'B', 'C']


## 22 — FSC post-hoc isotonic calibration -- sensitivity analysis only

**This variant is not used for any downstream decision** (thresholds, cost policies, or the primary
H2/H3 verdicts) -- it is fit and reported purely as a sensitivity analysis against the pre-specified
sigmoid variant from Section 21. Isotonic regression is flexible enough to reshape itself almost exactly
around whatever data it is fit on; even when fit on out-of-fold predictions (never the in-sample
predictions of a model fit on all of validation), it can still fit those OOF predictions' *own* noise
well enough to look artificially well-calibrated on the very data used to fit it, which is exactly the
failure mode Section 23 documents and the reason this notebook does not let an ECE comparison pick
isotonic as the primary variant. It is more prone to this with few validation positives -- flagged
explicitly below rather than assumed stable.

In [26]:
if "fsc" in ACTIVE_DATASETS:
    from sklearn.isotonic import IsotonicRegression

    MIN_VAL_POSITIVES_FOR_ISOTONIC = 20  # documented threshold below which isotonic is flagged as unstable

    y_val = R["fsc"]["y_val"]; oof_val_scores = R["fsc"]["oof_val_scores"]; test_scores = R["fsc"]["test_scores"]
    isotonic_calibrators, isotonic_oof_val_scores, isotonic_test_scores = {}, {}, {}
    n_pos = int(y_val.sum())
    for name in FEATURE_GROUPS:
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        iso.fit(oof_val_scores[name], y_val)
        isotonic_calibrators[name] = iso
        isotonic_oof_val_scores[name] = iso.predict(oof_val_scores[name])
        isotonic_test_scores[name] = iso.predict(test_scores[name])
        note = "stable" if n_pos >= MIN_VAL_POSITIVES_FOR_ISOTONIC else \
               f"CAUTION: only {n_pos} validation positives -- isotonic fit may be unstable/overfit"
        audit("fsc", f"isotonic_calibration_{name}", fit_on="validation (out-of-fold predictions)",
              frozen="IsotonicRegression on OOF risk score", evaluated_on="test (single pass)", note=note)

    if n_pos < MIN_VAL_POSITIVES_FOR_ISOTONIC:
        print(f"[fsc] CAUTION: only {n_pos} validation positives -- isotonic fit may be unstable/overfit")
    R["fsc"]["isotonic_calibrators"] = isotonic_calibrators
    R["fsc"]["isotonic_oof_val_scores"] = isotonic_oof_val_scores
    R["fsc"]["isotonic_test_scores"] = isotonic_test_scores
    print("[fsc] Isotonic calibration fit (on out-of-fold validation predictions) for models:",
          list(isotonic_calibrators.keys()))

[fsc] Isotonic calibration fit (on out-of-fold validation predictions) for models: ['A', 'B', 'C']


## 23 — FSC calibration method: pre-specified (sigmoid), isotonic reported as a sensitivity analysis only

**This notebook does not pick model C's calibration variant by whichever scores lowest on validation
ECE.** `best_variant_C` is fixed to `FSC_PRIMARY_CALIBRATION_VARIANT` (`"sigmoid"`, Section 09) --
decided before this cell computes a single number, not by sorting a metrics table.

**Why not just pick by lowest validation ECE?** An earlier revision of this notebook did exactly that,
using the out-of-fold (OOF) validation predictions from Sections 19-22 to avoid the in-sample-overfitting
problem. That fixed one leak but not a second, more subtle one: isotonic regression is flexible enough to
reshape itself almost exactly around whatever data it is fit on, so even fit on OOF predictions (rows the
underlying Random Forest never trained on) it can still land a near-zero ECE *on those same OOF rows*,
without that near-zero number generalizing. Concretely, on one run of this notebook the validation-side
comparison looked like this:

| Model C variant | Validation ECE (misleading) | Test ECE (what actually happened) |
|---|---|---|
| raw | 0.00604 | 0.00816 |
| sigmoid | 0.01357 | 0.01685 |
| isotonic | ≈0 | 0.01105 |

Isotonic "won" validation by a wide margin and then did **not** win on test -- exactly the overfitting
pattern scikit-learn's calibration documentation warns isotonic is prone to. Selecting the variant that
wins this comparison is itself a fitting decision that needs to generalize to unseen data, and this
notebook does not have a second, independent slice of FSC validation to make that selection on without
recursing into the same problem. Rather than build a nested/cross-fitted calibration-selection scheme on
top of an already-tiny FSC failure population (93 validation failures, 23 test failures), this notebook
takes the simpler, pre-registered route: **sigmoid is parametric and monotonic, so it cannot reshape
itself around its own fitting noise the way isotonic can; it is fixed as the primary method, and isotonic
is retained only as a labeled sensitivity analysis**, never feeding into a threshold, a cost policy, or
the primary H2/H3 verdict.

The cell below still computes the raw/sigmoid/isotonic table on OOF validation -- as a **diagnostic**,
not a selection mechanism -- and flags if isotonic's validation ECE looks suspiciously close to zero (the
exact symptom in the table above). FSC test is not read anywhere in this cell.

In [27]:
if "fsc" in ACTIVE_DATASETS:
    y_val = R["fsc"]["y_val"]
    oof_val_scores = R["fsc"]["oof_val_scores"]
    sigmoid_oof_val_scores = R["fsc"]["sigmoid_oof_val_scores"]
    isotonic_oof_val_scores = R["fsc"]["isotonic_oof_val_scores"]

    diagnostic_rows = []
    for name in ["B", "C"]:
        for variant, scores in [("raw", oof_val_scores[name]),
                                 ("sigmoid", sigmoid_oof_val_scores[name]),
                                 ("isotonic", isotonic_oof_val_scores[name])]:
            rep, _ = calibration_report(f"{name}_{variant}", y_val, scores)
            rep["model"] = name
            rep["calibration"] = variant
            diagnostic_rows.append(rep)

    calibration_metrics_val_df = pd.DataFrame(diagnostic_rows)[
        ["model", "calibration", "brier", "ece", "mce"]
    ].sort_values(["model", "calibration"])
    R["fsc"]["calibration_metrics_val_df"] = calibration_metrics_val_df

    # Pre-specified, not selected -- see markdown above.
    best_variant_C = FSC_PRIMARY_CALIBRATION_VARIANT
    R["fsc"]["best_variant_C"] = best_variant_C

    audit("fsc", "calibration_variant_prespecified", fit_on="-",
          frozen=f"best_variant_C = {best_variant_C} (pre-specified, not selected by any metric)",
          evaluated_on="-",
          note="Sigmoid pre-specified as the primary FSC calibration method before any calibration "
               "metric was computed; isotonic retained only as a sensitivity analysis. FSC test was not "
               "read in this step.")

    print("=== [fsc] calibration metrics on OUT-OF-FOLD VALIDATION (diagnostic only -- not a selection) ===")
    print(calibration_metrics_val_df.to_string(index=False))
    print(f"\n[fsc] Calibration method for model C is PRE-SPECIFIED as: {best_variant_C}")

    iso_val_ece = calibration_metrics_val_df.set_index(["model", "calibration"]).loc[("C", "isotonic"), "ece"]
    if iso_val_ece < 0.002:
        print(f"\n[fsc] DIAGNOSTIC: isotonic's validation ECE ({iso_val_ece:.5f}) is suspiciously close to "
              f"zero -- exactly the self-fit-overfitting symptom this section's markdown describes. This is "
              f"additional evidence for, not against, pre-specifying sigmoid rather than selecting by "
              f"validation ECE. Compare against isotonic's TEST ECE in Section 23b once it runs.")

    calibration_metrics_val_df.assign(dataset="fsc").to_csv(
        OUT_DIR / "fsc_voxintel_r_calibration_diagnostic_validation.csv", index=False)
    print(f"Saved: {OUT_DIR / 'fsc_voxintel_r_calibration_diagnostic_validation.csv'}")

=== [fsc] calibration metrics on OUT-OF-FOLD VALIDATION (diagnostic only -- not a selection) ===
model calibration    brier          ece          mce
    B    isotonic 0.025036 0.000000e+00 0.000000e+00
    B         raw 0.040153 4.861664e-02 6.274756e-01
    B     sigmoid 0.026501 5.353102e-03 7.493027e-01
    C    isotonic 0.021195 1.958379e-18 8.326673e-17
    C         raw 0.022139 6.042335e-03 7.166667e-01
    C     sigmoid 0.023888 1.357432e-02 5.473505e-01

[fsc] Calibration method for model C is PRE-SPECIFIED as: sigmoid

[fsc] DIAGNOSTIC: isotonic's validation ECE (0.00000) is suspiciously close to zero -- exactly the self-fit-overfitting symptom this section's markdown describes. This is additional evidence for, not against, pre-specifying sigmoid rather than selecting by validation ECE. Compare against isotonic's TEST ECE in Section 23b once it runs.
Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_calibration_diagnostic_validation.csv


## 23b — FSC calibration metrics: final test evaluation (reporting only)

Reports Brier score, ECE, and MCE on FSC's frozen test split for models B and C, across all three
calibration variants, for transparency -- including isotonic, so the sensitivity-analysis comparison from
Section 23 (validation) can be checked against test. `best_variant_C` (sigmoid) was pre-specified in
Section 23, before this cell runs; this cell evaluates test performance, it does not, and cannot, revise
that choice.

In [28]:
if "fsc" in ACTIVE_DATASETS:
    y_test = R["fsc"]["y_test"]
    test_scores = R["fsc"]["test_scores"]
    sigmoid_test_scores = R["fsc"]["sigmoid_test_scores"]
    isotonic_test_scores = R["fsc"]["isotonic_test_scores"]

    calibration_rows = []
    reliability_tables = {}
    for name in ["B", "C"]:
        for variant, scores in [("raw", test_scores[name]),
                                 ("sigmoid", sigmoid_test_scores[name]),
                                 ("isotonic", isotonic_test_scores[name])]:
            rep, reliab = calibration_report(f"{name}_{variant}", y_test, scores)
            rep["model"] = name
            rep["calibration"] = variant
            calibration_rows.append(rep)
            reliability_tables[f"{name}_{variant}"] = reliab

    calibration_metrics_df = pd.DataFrame(calibration_rows)[
        ["model", "calibration", "brier", "ece", "mce"]
    ].sort_values(["model", "calibration"])
    R["fsc"]["calibration_metrics_df"] = calibration_metrics_df
    R["fsc"]["reliability_tables"] = reliability_tables

    print("=== [fsc] calibration metrics on TEST (final reporting; sigmoid pre-specified in Section 23) ===")
    print(calibration_metrics_df.to_string(index=False))

    best_variant_C = R["fsc"]["best_variant_C"]
    best_row = calibration_metrics_df[(calibration_metrics_df["model"] == "C") &
                                       (calibration_metrics_df["calibration"] == best_variant_C)]
    print(f"\n[fsc] Test-set performance of the pre-specified primary variant (model C, {best_variant_C}):")
    print(best_row.to_string(index=False))

    iso_row = calibration_metrics_df[(calibration_metrics_df["model"] == "C") &
                                      (calibration_metrics_df["calibration"] == "isotonic")]
    print(f"\n[fsc] Sensitivity analysis -- isotonic's test-set performance (model C, not used downstream):")
    print(iso_row.to_string(index=False))

    calibration_metrics_df.assign(dataset="fsc").to_csv(
        OUT_DIR / "fsc_voxintel_r_final_calibration_metrics.csv", index=False)
    calibration_metrics_df.assign(dataset="fsc").to_csv(
        OUT_DIR / "notebook18_calibration_results.csv", index=False)

    reliability_long = pd.concat(
        [df.assign(variant=k, dataset="fsc") for k, df in reliability_tables.items()], ignore_index=True)
    reliability_long.to_csv(OUT_DIR / "fsc_voxintel_r_reliability_data.csv", index=False)
    reliability_long.to_csv(OUT_DIR / "notebook18_reliability_data.csv", index=False)

    print(f"\nSaved: {OUT_DIR / 'fsc_voxintel_r_final_calibration_metrics.csv'}")
    print(f"Saved: {OUT_DIR / 'fsc_voxintel_r_reliability_data.csv'}")

=== [fsc] calibration metrics on TEST (final reporting; sigmoid pre-specified in Section 23) ===
model calibration    brier      ece      mce
    B    isotonic 0.006901 0.019424 0.296590
    B         raw 0.023021 0.036854 0.630173
    B     sigmoid 0.007188 0.020816 0.617216
    C    isotonic 0.005588 0.011050 0.363636
    C         raw 0.005585 0.008162 0.263333
    C     sigmoid 0.006113 0.016854 0.662537

[fsc] Test-set performance of the pre-specified primary variant (model C, sigmoid):
model calibration    brier      ece      mce
    C     sigmoid 0.006113 0.016854 0.662537

[fsc] Sensitivity analysis -- isotonic's test-set performance (model C, not used downstream):
model calibration    brier     ece      mce
    C    isotonic 0.005588 0.01105 0.363636

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_final_calibration_metrics.csv
Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_reliability_data.csv


## 24 — FSC reliability diagrams

Plotted for model C (raw vs. the pre-specified sigmoid variant from Section 23) on FSC test. Bins with
fewer than 5 test examples are marked as low-confidence rather than plotted as if they were reliable.

In [29]:
if "fsc" in ACTIVE_DATASETS:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    reliability_tables = R["fsc"]["reliability_tables"]
    y_test = R["fsc"]["y_test"]
    best_variant_C = R["fsc"]["best_variant_C"]  # pre-specified in Section 23, not selected on any metric

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, variant in zip(axes, ["raw", best_variant_C]):
        reliab = reliability_tables[f"C_{variant}"]
        ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
        plotted = reliab.dropna(subset=["mean_predicted", "observed_freq"])
        low_n = plotted[plotted["count"] < 5]
        ok_n = plotted[plotted["count"] >= 5]
        ax.scatter(ok_n["mean_predicted"], ok_n["observed_freq"],
                   s=ok_n["count"] * 3 + 20, label="bin (n>=5)", color="tab:blue")
        ax.scatter(low_n["mean_predicted"], low_n["observed_freq"],
                   s=30, marker="x", label="bin (n<5, low confidence)", color="tab:red")
        ax.set_xlabel("Mean predicted risk")
        ax.set_ylabel("Observed failure frequency")
        ax.set_title(f"Model C -- {variant}")
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.legend(fontsize=8)
    fig.suptitle(f"Reliability diagrams (FSC test, n={len(y_test)}, {int(y_test.sum())} positives)")
    fig.tight_layout()
    reliability_plot_path = OUT_DIR / "fsc_voxintel_r_reliability_diagram.png"
    fig.savefig(reliability_plot_path, dpi=150)
    plt.close(fig)
    print(f"Saved: {reliability_plot_path}")

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_reliability_diagram.png


## 25 — FSC selective prediction setup

**Decision rule:**

```
if P(failure) < tau:  EXECUTE
if P(failure) >= tau: CLARIFY / ABSTAIN
```

Four signals are compared: always-execute, ASR confidence, intent confidence, and the pre-specified
sigmoid-calibrated VoxIntel-R combined risk (model C, Section 23). Direction matters and is made explicit
for each. The validation-side risk used here is the **out-of-fold** calibrated risk, not an in-sample
score -- this is what Section 26 selects a threshold on.

In [30]:
if "fsc" in ACTIVE_DATASETS:
    val_df = loaded["fsc"]["val_df"]; test_df = loaded["fsc"]["test_df"]
    best_variant_C = R["fsc"]["best_variant_C"]
    oof_val_scores = R["fsc"]["oof_val_scores"]; test_scores = R["fsc"]["test_scores"]
    sigmoid_oof_val_scores = R["fsc"]["sigmoid_oof_val_scores"]; sigmoid_test_scores = R["fsc"]["sigmoid_test_scores"]
    isotonic_oof_val_scores = R["fsc"]["isotonic_oof_val_scores"]; isotonic_test_scores = R["fsc"]["isotonic_test_scores"]

    risk_val_C = {"raw": oof_val_scores["C"], "sigmoid": sigmoid_oof_val_scores["C"],
                  "isotonic": isotonic_oof_val_scores["C"]}[best_variant_C]
    risk_test_C = {"raw": test_scores["C"], "sigmoid": sigmoid_test_scores["C"],
                   "isotonic": isotonic_test_scores["C"]}[best_variant_C]
    R["fsc"]["risk_val_C"] = risk_val_C
    R["fsc"]["risk_test_C"] = risk_test_C

    FSC_SIGNALS = {
        "always_execute": None,
        "asr_confidence": {"val": val_df["asr_mean_confidence"].values, "test": test_df["asr_mean_confidence"].values, "higher_is_safer": True},
        "intent_confidence": {"val": val_df["intent_confidence"].values, "test": test_df["intent_confidence"].values, "higher_is_safer": True},
        "voxintel_r_risk_C": {"val": risk_val_C, "test": risk_test_C, "higher_is_safer": False},
    }
    R["fsc"]["SIGNALS"] = FSC_SIGNALS
    print(f"[fsc] Signals configured: {list(FSC_SIGNALS.keys())} "
          f"(risk calibration variant: {best_variant_C}, validation-side = out-of-fold)")

[fsc] Signals configured: ['always_execute', 'asr_confidence', 'intent_confidence', 'voxintel_r_risk_C'] (risk calibration variant: sigmoid, validation-side = out-of-fold)


## 26 — FSC validation threshold selection

A single representative operating point is selected **on FSC validation only**, at a target coverage of
90%. This threshold is then frozen and applied once to FSC test. The full risk-coverage curve (Section 27)
is threshold-free and reported alongside it.

In [31]:
if "fsc" in ACTIVE_DATASETS:
    SIGNALS = R["fsc"]["SIGNALS"]
    frozen_thresholds = {}
    for signal, cfg in SIGNALS.items():
        if signal == "always_execute":
            frozen_thresholds[signal] = None
            continue
        tau = threshold_at_target_coverage(cfg["val"], cfg["higher_is_safer"], TARGET_COVERAGE)
        frozen_thresholds[signal] = tau
        audit("fsc", f"selective_threshold_{signal}", fit_on="validation",
              frozen=f"tau={tau:.6f} (target coverage={TARGET_COVERAGE})", evaluated_on="test (single pass)")
    R["fsc"]["frozen_thresholds"] = frozen_thresholds
    print(f"[fsc] Frozen thresholds (validation-selected, target coverage={TARGET_COVERAGE}):")
    for k, v in frozen_thresholds.items():
        print(f"    {k}: {v}")

[fsc] Frozen thresholds (validation-selected, target coverage=0.9):
    always_execute: None
    asr_confidence: 0.9886168539524078
    intent_confidence: 0.9950779676437378
    voxintel_r_risk_C: 0.07355607737657767


## 27 — FSC test risk-coverage evaluation and baseline comparison

Full risk-coverage curves computed on FSC's own frozen test split for every signal (threshold-free), plus
the single frozen 90%-coverage operating point from Section 26 applied to test. The H3 verdict
(Section 33) is decided from `comparison_table` below.

In [32]:
if "fsc" in ACTIVE_DATASETS:
    import matplotlib.pyplot as plt

    y_test = R["fsc"]["y_test"]; SIGNALS = R["fsc"]["SIGNALS"]; frozen_thresholds = R["fsc"]["frozen_thresholds"]

    curves, operating_points, aurc_rows = {}, [], []
    for signal, cfg in SIGNALS.items():
        if signal == "always_execute":
            curve = pd.DataFrame({"coverage": [1.0], "selective_risk": [y_test.mean()],
                                   "wrong_execution_rate": [y_test.mean()], "clarification_rate": [0.0]})
            op = {"coverage": 1.0, "selective_risk": y_test.mean(),
                  "wrong_execution_rate": y_test.mean(), "clarification_rate": 0.0}
            aurc = y_test.mean()
        else:
            curve = risk_coverage_curve(y_test, cfg["test"], cfg["higher_is_safer"])
            op = apply_frozen_threshold(y_test, cfg["test"], cfg["higher_is_safer"], frozen_thresholds[signal])
            aurc = area_under_risk_coverage(curve)
        curves[signal] = curve
        op["signal"] = signal
        operating_points.append(op)
        aurc_rows.append({"signal": signal, "aurc": aurc})

    operating_points_df = pd.DataFrame(operating_points)[
        ["signal", "coverage", "selective_risk", "wrong_execution_rate", "clarification_rate"]
    ]
    aurc_df = pd.DataFrame(aurc_rows)
    R["fsc"]["curves"] = curves
    R["fsc"]["operating_points_df"] = operating_points_df
    R["fsc"]["aurc_df"] = aurc_df

    print(f"=== [fsc] operating points at target coverage={TARGET_COVERAGE} (frozen thresholds, test) ===")
    print(operating_points_df.to_string(index=False))
    print("\n[fsc] Area under risk-coverage curve (lower = better):")
    print(aurc_df.to_string(index=False))

    risk_coverage_long = pd.concat([c.assign(signal=s) for s, c in curves.items()], ignore_index=True)
    risk_coverage_long.to_csv(OUT_DIR / "fsc_voxintel_r_risk_coverage.csv", index=False)
    comparison_table = operating_points_df.merge(aurc_df, on="signal").sort_values("aurc")
    R["fsc"]["comparison_table"] = comparison_table
    comparison_table.to_csv(OUT_DIR / "fsc_voxintel_r_selective_results.csv", index=False)
    comparison_table.to_csv(OUT_DIR / "notebook18_fsc_test_results.csv", index=False)

    print("\n=== [fsc] comparison table (sorted by AURC, lower=better) ===")
    print(comparison_table.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 5))
    for signal, curve in curves.items():
        if signal == "always_execute":
            ax.scatter([1.0], [y_test.mean()], label="always_execute", marker="x", color="black", zorder=5)
        else:
            ax.plot(curve["coverage"], curve["selective_risk"], label=signal)
    ax.set_xlabel("Coverage (fraction executed)")
    ax.set_ylabel("Selective risk (failure rate among executed)")
    ax.set_title("Risk-coverage curves (FSC test)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    rc_plot_path = OUT_DIR / "fsc_voxintel_r_risk_coverage.png"
    fig.savefig(rc_plot_path, dpi=150)
    plt.close(fig)
    print(f"\nSaved: {OUT_DIR / 'fsc_voxintel_r_risk_coverage.csv'}, "
          f"{OUT_DIR / 'fsc_voxintel_r_selective_results.csv'}, {rc_plot_path}")

=== [fsc] operating points at target coverage=0.9 (frozen thresholds, test) ===
           signal  coverage  selective_risk  wrong_execution_rate  clarification_rate
   always_execute  1.000000        0.006064              0.006064            0.000000
   asr_confidence  0.945162        0.004184              0.003955            0.054838
intent_confidence  0.904825        0.004079              0.003691            0.095175
voxintel_r_risk_C  0.943317        0.003354              0.003164            0.056683

[fsc] Area under risk-coverage curve (lower = better):
           signal     aurc
   always_execute 0.006064
   asr_confidence 0.004590
intent_confidence 0.002840
voxintel_r_risk_C 0.003103

=== [fsc] comparison table (sorted by AURC, lower=better) ===
           signal  coverage  selective_risk  wrong_execution_rate  clarification_rate     aurc
intent_confidence  0.904825        0.004079              0.003691            0.095175 0.002840
voxintel_r_risk_C  0.943317        0.003354   

## 27b — FSC statistical uncertainty for H3 (paired bootstrap ΔAURC)

Section 27's point estimates show VoxIntel-R risk beating every baseline's AURC on FSC test -- but FSC
test has only 23 positive (failure) cases out of 3,793, and the margin over the next-best baseline can be
as small as a few thousandths of a point. A point estimate alone cannot say whether that margin reflects
a real effect or noise. This section runs the **same AURC-bootstrap machinery already used for SLURP**
(`run_aurc_bootstrap_suite`, Section 12/15) on FSC test: for each of 2,000 resamples, the entire
risk-coverage curve and AURC are recomputed for VoxIntel-R risk and for each baseline, and the paired
delta is recorded. Section 33's H3 verdict is decided from this table together with Section 27's point
estimates, using the same shared decision rule (`verdict_h3`, Section 12) applied to SLURP -- including
the possibility of `INCONCLUSIVE` when every point estimate favors the risk score but none of those wins
are statistically distinguishable from zero.

In [33]:
if "fsc" in ACTIVE_DATASETS:
    y_test = R["fsc"]["y_test"]
    SIGNALS = R["fsc"]["SIGNALS"]
    risk_test_C = SIGNALS["voxintel_r_risk_C"]["test"]
    rng = np.random.default_rng(RANDOM_SEED)

    fsc_baseline_signals = {
        "always_execute": {"scores": None, "higher_is_safer": True},
        "asr_confidence": {"scores": SIGNALS["asr_confidence"]["test"],
                            "higher_is_safer": SIGNALS["asr_confidence"]["higher_is_safer"]},
        "intent_confidence": {"scores": SIGNALS["intent_confidence"]["test"],
                               "higher_is_safer": SIGNALS["intent_confidence"]["higher_is_safer"]},
    }
    fsc_h3_bootstrap_df = run_aurc_bootstrap_suite(
        y_test, risk_test_C, False, "voxintel_r_risk_C", fsc_baseline_signals, rng)
    R["fsc"]["h3_bootstrap_df"] = fsc_h3_bootstrap_df
    fsc_h3_bootstrap_df.to_csv(OUT_DIR / "fsc_voxintel_r_bootstrap_h3.csv", index=False)

    print(f"=== [fsc] paired bootstrap AURC deltas (n_test={len(y_test)}, "
          f"{int(y_test.sum())} positives; negative delta = risk is better) ===")
    print(fsc_h3_bootstrap_df.to_string(index=False))

    if not fsc_h3_bootstrap_df["favors_risk"].any():
        print("\n[fsc] NOTE: none of the AURC-bootstrap comparisons are both statistically significant and "
              "in favor of VoxIntel-R risk. With only 23 FSC test positives, a point-estimate win this "
              "small is exactly the situation this bootstrap is designed to catch -- see the H3 decision "
              "rule in Section 33, which can return INCONCLUSIVE here even if Section 27's point estimates "
              "all favor VoxIntel-R risk.")

    print(f"\nSaved: {OUT_DIR / 'fsc_voxintel_r_bootstrap_h3.csv'}")

=== [fsc] paired bootstrap AURC deltas (n_test=3793, 23 positives; negative delta = risk is better) ===
                            comparison metric  delta_mean     ci_lo     ci_hi  significant  favors_risk
   voxintel_r_risk_C vs always_execute   aurc   -0.003127 -0.005638 -0.000473         True         True
   voxintel_r_risk_C vs asr_confidence   aurc   -0.001710 -0.003809  0.000533        False        False
voxintel_r_risk_C vs intent_confidence   aurc    0.000039 -0.001439  0.001901        False        False

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_bootstrap_h3.csv


## 28 — FSC cost model design

**SIMULATED / ASSIGNED COST ASSUMPTIONS.** FSC contains no real-world severity ground truth (no
banking/medical/irreversible-action labels). Everything from here through Section 30 is a sensitivity
analysis over *assumed* cost regimes, run only on FSC. **This section is not run on SLURP** (see
Section 16): a cost-*optimal* threshold needs to be selected against labels, and SLURP has no independent
split to select one on without evaluating it against the same labels used to pick it.

**Because severity is simulated (hash-assigned) whenever `SEVERITY_OVERRIDES` does not cover every
observed intent, Section 33 reports H4 as `INCONCLUSIVE` by default** -- the cost-sensitivity numbers
below are kept and reported as an illustrative sensitivity analysis ("under these assumed costs, the
policy behaves like this"), but a win-count computed against hash-assigned severity is not evidence for or
against the H4 hypothesis itself. Supplying a real per-intent mapping in `SEVERITY_OVERRIDES` (covering
every unique intent value) turns H4 back into a genuine, gradable hypothesis test.

In [34]:
if "fsc" in ACTIVE_DATASETS:
    SEVERITY_TIERS = ["LOW", "MEDIUM", "HIGH"]
    # Edit to encode a real severity mapping for FSC's intent taxonomy, e.g.:
    # SEVERITY_OVERRIDES = {"increase_volume": "LOW", "transfer_money": "HIGH", ...}
    # H4 (Section 33) only becomes a genuine hypothesis test once EVERY unique intent value
    # encountered has an explicit entry here -- see severity_is_simulated below.
    SEVERITY_OVERRIDES = {}

    def _hash_bucket(value, n_buckets=3, seed=RANDOM_SEED):
        h = hashlib.sha256(f"{seed}:{value}".encode()).hexdigest()
        return int(h, 16) % n_buckets

    def assign_severity(raw_df, val_df, test_df, id_col, target_col):
        intent_col = COLUMN_MAP.get("intent_label")
        if not intent_col:
            auto = [c for c in raw_df.columns
                    if "intent" in c.lower() and c not in ALLOWED_FEATURE_COLUMNS and c != target_col]
            intent_col = auto[0] if auto else None

        if intent_col and intent_col in raw_df.columns:
            unique_intents = sorted(raw_df[intent_col].astype(str).unique())
            n_overridden = sum(1 for i in unique_intents if i in SEVERITY_OVERRIDES)
            severity_map = {i: SEVERITY_OVERRIDES.get(i, SEVERITY_TIERS[_hash_bucket(f"fsc:{i}")])
                             for i in unique_intents}
            val_sev = val_df[intent_col].astype(str).map(severity_map).values
            test_sev = test_df[intent_col].astype(str).map(severity_map).values
            is_simulated = n_overridden < len(unique_intents)
            if is_simulated:
                source_note = (f"{n_overridden}/{len(unique_intents)} unique values of '{intent_col}' have "
                                f"an explicit SEVERITY_OVERRIDES entry; the remaining "
                                f"{len(unique_intents) - n_overridden} fall back to deterministic hash "
                                f"bucketing (SIMULATED).")
            else:
                source_note = (f"All {len(unique_intents)} unique values of '{intent_col}' have an "
                                f"explicit SEVERITY_OVERRIDES entry (real mapping, not simulated).")
        else:
            id_series_val = val_df[id_col] if id_col in val_df.columns else pd.Series(val_df.index)
            id_series_test = test_df[id_col] if id_col in test_df.columns else pd.Series(test_df.index)
            val_sev = np.array([SEVERITY_TIERS[_hash_bucket(f"fsc:{v}")] for v in id_series_val])
            test_sev = np.array([SEVERITY_TIERS[_hash_bucket(f"fsc:{v}")] for v in id_series_test])
            is_simulated = True
            source_note = "no intent-identity column found -- per-utterance hash fallback (SIMULATED)"
        return val_sev, test_sev, source_note, is_simulated

    val_severity, test_severity, severity_source_note, severity_is_simulated = assign_severity(
        loaded["fsc"]["raw_df"], loaded["fsc"]["val_df"], loaded["fsc"]["test_df"],
        COLUMN_MAP.get("id", "utterance_id"), R["fsc"]["target_col"])
    R["fsc"]["val_severity"] = val_severity
    R["fsc"]["test_severity"] = test_severity
    R["fsc"]["severity_source_note"] = severity_source_note
    R["fsc"]["severity_is_simulated"] = severity_is_simulated

    print("=== [fsc] severity assignment ===")
    print("Source:", severity_source_note)
    print("Validation severity tier counts:"); print(pd.Series(val_severity).value_counts())
    print("Test severity tier counts:"); print(pd.Series(test_severity).value_counts())
    audit("fsc", "severity_assignment", fit_on="-", frozen="-", evaluated_on="-", note=severity_source_note)

    if severity_is_simulated:
        print("\n[fsc] NOTE: severity is SIMULATED for at least one intent value. Per Section 33, H4 will "
              "be reported as INCONCLUSIVE (sensitivity analysis only) rather than SUPPORTED/NOT SUPPORTED "
              "-- a win-count under hash-assigned severity is not a valid test of a cost-sensitivity "
              "hypothesis. Populate SEVERITY_OVERRIDES above with a real mapping covering every intent to "
              "enable a genuine H4 test.")

=== [fsc] severity assignment ===
Source: no intent-identity column found -- per-utterance hash fallback (SIMULATED)
Validation severity tier counts:
LOW       1055
HIGH      1052
MEDIUM    1011
Name: count, dtype: int64
Test severity tier counts:
LOW       1295
HIGH      1252
MEDIUM    1246
Name: count, dtype: int64

[fsc] NOTE: severity is SIMULATED for at least one intent value. Per Section 33, H4 will be reported as INCONCLUSIVE (sensitivity analysis only) rather than SUPPORTED/NOT SUPPORTED -- a win-count under hash-assigned severity is not a valid test of a cost-sensitivity hypothesis. Populate SEVERITY_OVERRIDES above with a real mapping covering every intent to enable a genuine H4 test.


## 29 — FSC severity/cost sensitivity analysis

**SIMULATED / ASSIGNED COST ASSUMPTIONS.** Multiple cost regimes are run -- not one cherry-picked matrix --
so conclusions are read from the pattern across regimes, not a single number.

In [35]:
if "fsc" in ACTIVE_DATASETS:
    COST_REGIMES = {
        "regime_1": {"LOW": 1, "MEDIUM": 2, "HIGH": 5},
        "regime_2": {"LOW": 1, "MEDIUM": 5, "HIGH": 20},
        "regime_3": {"LOW": 1, "MEDIUM": 10, "HIGH": 50},
    }
    CLARIFICATION_COSTS = [0.1, 0.5, 1.0]

    def wrong_cost_array(severity, regime):
        return np.array([regime[s] for s in severity], dtype=float)

    print("Cost regimes (SIMULATED / ASSIGNED, FSC only):")
    for r, cfg in COST_REGIMES.items():
        print(f"  {r}: {cfg}")
    print("Clarification costs tested:", CLARIFICATION_COSTS)

    R["fsc"]["val_wrong_cost"] = {r: wrong_cost_array(R["fsc"]["val_severity"], cfg) for r, cfg in COST_REGIMES.items()}
    R["fsc"]["test_wrong_cost"] = {r: wrong_cost_array(R["fsc"]["test_severity"], cfg) for r, cfg in COST_REGIMES.items()}

Cost regimes (SIMULATED / ASSIGNED, FSC only):
  regime_1: {'LOW': 1, 'MEDIUM': 2, 'HIGH': 5}
  regime_2: {'LOW': 1, 'MEDIUM': 5, 'HIGH': 20}
  regime_3: {'LOW': 1, 'MEDIUM': 10, 'HIGH': 50}
Clarification costs tested: [0.1, 0.5, 1.0]


## 30 — FSC cost-sensitive policy evaluation

Five policies compared on FSC: always-execute, ASR-confidence threshold, intent-confidence threshold, and
a Bayes-rule severity-aware VoxIntel-R policy, each cost-optimal threshold selected on **FSC validation
only** per regime, frozen, then evaluated once on FSC test.

In [36]:
if "fsc" in ACTIVE_DATASETS:
    import matplotlib.pyplot as plt

    def expected_cost(y_true, execute_mask, wrong_cost, clarification_cost):
        wrong = execute_mask & (y_true == 1)
        clarify = ~execute_mask
        total = wrong_cost[wrong].sum() + clarification_cost * clarify.sum()
        return total / len(y_true)

    def _cost_optimal_threshold(y_val, score, higher_is_safer, wrong_cost_val, clarification_cost, n_grid=200):
        lo, hi = float(np.min(score)), float(np.max(score))
        grid = np.linspace(lo, hi, n_grid)
        best_tau, best_cost = None, np.inf
        for tau in grid:
            execute = (score >= tau) if higher_is_safer else (score <= tau)
            c = expected_cost(y_val, execute, wrong_cost_val, clarification_cost)
            if c < best_cost:
                best_cost, best_tau = c, tau
        return best_tau

    val_df = loaded["fsc"]["val_df"]; test_df = loaded["fsc"]["test_df"]
    y_val = R["fsc"]["y_val"]; y_test = R["fsc"]["y_test"]
    risk_val_C = R["fsc"]["risk_val_C"]; risk_test_C = R["fsc"]["risk_test_C"]
    val_wrong_cost = R["fsc"]["val_wrong_cost"]; test_wrong_cost = R["fsc"]["test_wrong_cost"]

    uniform_signals = {
        "asr_confidence_threshold": {"val": val_df["asr_mean_confidence"].values, "test": test_df["asr_mean_confidence"].values, "higher_is_safer": True},
        "intent_confidence_threshold": {"val": val_df["intent_confidence"].values, "test": test_df["intent_confidence"].values, "higher_is_safer": True},
        "voxintel_r_uniform": {"val": risk_val_C, "test": risk_test_C, "higher_is_safer": False},
    }

    policy_rows = []
    for regime_name, regime_cfg in COST_REGIMES.items():
        wc_val = val_wrong_cost[regime_name]; wc_test = test_wrong_cost[regime_name]
        for cc in CLARIFICATION_COSTS:
            exec_mask = np.ones(len(y_test), dtype=bool)
            cost = expected_cost(y_test, exec_mask, wc_test, cc)
            policy_rows.append({"regime": regime_name, "clarification_cost": cc,
                                 "policy": "always_execute", "expected_cost": cost})

            for pname, cfg in uniform_signals.items():
                tau = _cost_optimal_threshold(y_val, cfg["val"], cfg["higher_is_safer"], wc_val, cc)
                execute = (cfg["test"] >= tau) if cfg["higher_is_safer"] else (cfg["test"] <= tau)
                cost = expected_cost(y_test, execute, wc_test, cc)
                policy_rows.append({"regime": regime_name, "clarification_cost": cc,
                                     "policy": pname, "expected_cost": cost})

            execute_bayes = (risk_test_C * wc_test) <= cc
            cost = expected_cost(y_test, execute_bayes, wc_test, cc)
            policy_rows.append({"regime": regime_name, "clarification_cost": cc,
                                 "policy": "voxintel_r_severity_aware", "expected_cost": cost})

        audit("fsc", f"cost_policy_thresholds_{regime_name}",
              fit_on="validation (grid search per clarification cost)",
              frozen="thresholds above; severity-aware policy uses analytic Bayes rule (no fitting)",
              evaluated_on="test (single pass per policy per cost combination)")

    cost_sensitive_df = pd.DataFrame(policy_rows)
    R["fsc"]["cost_sensitive_df"] = cost_sensitive_df
    cost_sensitive_df.to_csv(OUT_DIR / "fsc_voxintel_r_cost_sensitive_results.csv", index=False)

    sensitivity_pivot = cost_sensitive_df.pivot_table(
        index=["regime", "clarification_cost"], columns="policy", values="expected_cost").round(4)
    R["fsc"]["sensitivity_pivot"] = sensitivity_pivot
    sensitivity_pivot.to_csv(OUT_DIR / "fsc_voxintel_r_cost_sensitivity.csv")
    print("=== [fsc] cost sensitivity ===")
    print(sensitivity_pivot.to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    for policy in cost_sensitive_df["policy"].unique():
        d = cost_sensitive_df[cost_sensitive_df["policy"] == policy].sort_values(["regime", "clarification_cost"])
        xlabels = d["regime"] + "_cc" + d["clarification_cost"].astype(str)
        ax.plot(xlabels, d["expected_cost"], marker="o", label=policy, alpha=0.8)
    ax.set_xticks(range(len(xlabels)))
    ax.set_xticklabels(xlabels, rotation=60, ha="right", fontsize=7)
    ax.set_ylabel("Expected cost per utterance (SIMULATED)")
    ax.set_title("Cost-sensitivity analysis across regimes (FSC test)")
    ax.legend(fontsize=7)
    fig.tight_layout()
    cost_plot_path = OUT_DIR / "fsc_voxintel_r_cost_sensitivity.png"
    fig.savefig(cost_plot_path, dpi=150)
    plt.close(fig)
    print(f"Saved: {OUT_DIR / 'fsc_voxintel_r_cost_sensitive_results.csv'}, "
          f"{OUT_DIR / 'fsc_voxintel_r_cost_sensitivity.csv'}, {cost_plot_path}")

    win_counts = cost_sensitive_df.loc[
        cost_sensitive_df.groupby(["regime", "clarification_cost"])["expected_cost"].idxmin()
    ]["policy"].value_counts()
    R["fsc"]["win_counts"] = win_counts
    print("\n[fsc] How often each policy achieves the lowest expected cost (of 9 regime x cc combos):")
    print(win_counts.to_string())

=== [fsc] cost sensitivity ===
policy                       always_execute  asr_confidence_threshold  intent_confidence_threshold  voxintel_r_severity_aware  voxintel_r_uniform
regime   clarification_cost                                                                                                                      
regime_1 0.1                         0.0192                    0.0236                       0.0134                     0.0240              0.0250
         0.5                         0.0192                    0.0167                       0.0142                     0.0177              0.0182
         1.0                         0.0192                    0.0158                       0.0158                     0.0161              0.0163
regime_2 0.1                         0.0667                    0.0643                       0.0996                     0.0419              0.0517
         0.5                         0.0667                    0.0531                       0

## 31 — FSC feature / model ablation

Because H1 failed on FSC, this narrows in on which parts of each feature group carry the signal. Each
subset follows the same validation-fit -> freeze -> single-test-evaluation discipline as Section 19.

In [37]:
if "fsc" in ACTIVE_DATASETS:
    ABLATION_GROUPS = {
        "B_confidence_only": ["intent_confidence"],
        "B_entropy_only": ["intent_entropy"],
        "B_margin_only": ["intent_margin"],
        "C_without_duration": [f for f in COMBINED_FEATURES if f != "audio_duration"],
        "C_without_asr_entropy": [f for f in COMBINED_FEATURES
                                   if f not in {"asr_mean_entropy", "asr_max_entropy", "asr_std_entropy"}],
        "C_without_asr_confidence": [f for f in COMBINED_FEATURES
                                      if f not in {"asr_mean_confidence", "asr_min_confidence",
                                                   "asr_std_confidence", "asr_median_confidence"}],
    }

    val_df = loaded["fsc"]["val_df"]; test_df = loaded["fsc"]["test_df"]
    y_val = R["fsc"]["y_val"]; y_test = R["fsc"]["y_test"]; test_scores = R["fsc"]["test_scores"]

    ablation_test_scores, ablation_rows = {}, []
    for name, feats in {**FEATURE_GROUPS, **ABLATION_GROUPS}.items():
        if name in test_scores:
            p_test = test_scores[name]
        else:
            X_val = val_df[feats].values; X_test = test_df[feats].values
            model = make_rf(); model.fit(X_val, y_val)
            p_test = model.predict_proba(X_test)[:, 1]
            ablation_test_scores[name] = p_test
            audit("fsc", f"ablation_{name}", fit_on="validation (full refit)",
                  frozen=f"RandomForestClassifier over {feats}", evaluated_on="test (single pass)")
        ablation_rows.append({
            "subset": name, "n_features": len(feats),
            "roc_auc": roc_auc_score(y_test, p_test),
            "pr_auc": average_precision_score(y_test, p_test),
            "brier": brier_score_loss(y_test, p_test),
        })

    ablation_df = pd.DataFrame(ablation_rows)
    R["fsc"]["ablation_df"] = ablation_df
    R["fsc"]["ablation_test_scores"] = ablation_test_scores
    ablation_df.to_csv(OUT_DIR / "fsc_voxintel_r_feature_ablation.csv", index=False)
    print("=== [fsc] feature ablation (test) ===")
    print(ablation_df.to_string(index=False))

=== [fsc] feature ablation (test) ===
                  subset  n_features  roc_auc   pr_auc    brier
                       A           9 0.674824 0.154430 0.006176
                       B           3 0.731300 0.170698 0.023021
                       C          12 0.752779 0.228462 0.005585
       B_confidence_only           1 0.682811 0.106228 0.041742
          B_entropy_only           1 0.729085 0.206285 0.023187
           B_margin_only           1 0.717293 0.172479 0.041307
      C_without_duration          11 0.750450 0.215438 0.005631
   C_without_asr_entropy           9 0.736345 0.235657 0.005566
C_without_asr_confidence           8 0.741131 0.262295 0.005623


## 32 — FSC statistical uncertainty (paired bootstrap)

With few positive test cases, point estimates alone are not enough. Paired bootstrap (resampling test
rows jointly across compared scores, 2,000 resamples) gives 95% confidence intervals for the ablation
deltas against each parent model.

In [38]:
if "fsc" in ACTIVE_DATASETS:
    bootstrap_pairs = [
        ("B_confidence_only", "B"), ("B_entropy_only", "B"), ("B_margin_only", "B"),
        ("C_without_duration", "C"), ("C_without_asr_entropy", "C"), ("C_without_asr_confidence", "C"),
    ]

    y_test = R["fsc"]["y_test"]
    all_ablation_scores = {**R["fsc"]["test_scores"], **R["fsc"]["ablation_test_scores"]}
    rng = np.random.default_rng(RANDOM_SEED)

    bootstrap_rows = []
    for subset, parent in bootstrap_pairs:
        for metric_name, metric_fn in [("roc_auc", roc_auc_score), ("pr_auc", average_precision_score),
                                        ("brier", brier_score_loss)]:
            mean_d, lo, hi = paired_bootstrap_delta(y_test, all_ablation_scores[parent],
                                                     all_ablation_scores[subset], metric_fn, rng)
            bootstrap_rows.append({"subset": subset, "parent": parent, "metric": metric_name,
                                    "delta_mean": mean_d, "ci_lo": lo, "ci_hi": hi,
                                    "significant": not (lo <= 0 <= hi)})

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    R["fsc"]["bootstrap_df"] = bootstrap_df
    bootstrap_df.to_csv(OUT_DIR / "fsc_voxintel_r_bootstrap_ablation.csv", index=False)
    print("=== [fsc] paired bootstrap ablation deltas ===")
    print(bootstrap_df.to_string(index=False))
    n_sig = int(bootstrap_df["significant"].sum())
    print(f"[fsc] {n_sig} / {len(bootstrap_df)} ablation deltas have a 95% CI excluding zero "
          f"({int(y_test.sum())} positive test cases -- treat single significant deltas with caution).")

=== [fsc] paired bootstrap ablation deltas ===
                  subset parent  metric  delta_mean     ci_lo     ci_hi  significant
       B_confidence_only      B roc_auc   -0.048282 -0.104591 -0.016546         True
       B_confidence_only      B  pr_auc   -0.069598 -0.189329  0.016801        False
       B_confidence_only      B   brier    0.018706  0.016210  0.021075         True
          B_entropy_only      B roc_auc   -0.002277 -0.009373  0.001459        False
          B_entropy_only      B  pr_auc    0.030707 -0.098204  0.160232        False
          B_entropy_only      B   brier    0.000163 -0.000192  0.000572        False
           B_margin_only      B roc_auc   -0.014000 -0.025377 -0.003569         True
           B_margin_only      B  pr_auc   -0.000352 -0.048499  0.051657        False
           B_margin_only      B   brier    0.018298  0.015987  0.020766         True
      C_without_duration      C roc_auc   -0.002338 -0.044273  0.050413        False
      C_without_du

## 33 — Final H2/H3/H4 verdicts (per dataset)

**Decision criteria, stated before reading the numbers below** (so the verdict is not rationalized after
the fact):

**H2 — Calibration** (ECE is the calibration-selection/primary metric; MCE is evaluated as a secondary
reliability diagnostic, not a selection criterion -- FSC's calibration method itself was pre-specified in
Section 23 and never selected by either metric, so there is no selection step for MCE to be inconsistent
with here. Evaluated on held-out test for FSC / on the full evaluation population for SLURP):
- **SUPPORTED** if ECE < 0.05 and MCE < 0.20
- **PARTIALLY SUPPORTED** if ECE < 0.10 (this can still fail on MCE -- e.g. a good average calibration
  with one badly-off bin -- which is exactly why MCE is reported alongside ECE rather than folded into a
  single pass/fail number)
- **NOT SUPPORTED** otherwise

**H3 — Selective prediction** (AURC, lower = better -- this is the quantity the point estimate in
Section 14/27 *and* the significance test in Section 15/27b are both computed on, for both datasets, so
the point estimate and the significance test always ask the same question. A bootstrap 95% CI on ΔAURC
must exclude zero, and favor the risk score, for a point-estimate "win" to count as more than noise --
this matters most for FSC, where only 23 test failures make small point-estimate margins easy to
over-read):
- **SUPPORTED** if VoxIntel-R risk beats always-execute, ASR-confidence, and intent-confidence on point
  estimate, AND at least one of those wins is statistically significant (bootstrap CI excludes zero and
  favors the risk score)
- **INCONCLUSIVE** if VoxIntel-R risk beats all three on point estimate, but none of those wins are
  statistically distinguishable from zero -- a real-looking ranking that the data cannot actually support
- **PARTIALLY SUPPORTED** if it beats always-execute only
- **NOT SUPPORTED** if it does not even beat always-execute

**H4 — Cost-sensitive decision making** (FSC only; win-count of the severity-aware policy across 9
regime x clarification-cost combinations):
- **INCONCLUSIVE** whenever severity is simulated/hash-assigned (`R["fsc"]["severity_is_simulated"]` is
  `True`, Section 28) -- a win-count computed against hash-assigned severity does not test a
  cost-sensitivity hypothesis, it only illustrates how the policy behaves under an assumed cost structure.
  This is the default until a real per-intent severity mapping is supplied.
- Only once severity is a real, non-simulated mapping:
  - **SUPPORTED** if the severity-aware policy wins >= 5 / 9
  - **PARTIALLY SUPPORTED** if it wins >= 2 / 9
  - **NOT SUPPORTED** otherwise
- SLURP: **INCONCLUSIVE** by protocol (Section 16) -- not a result, a stated non-result.

In [39]:
def verdict_h2_from_report(report_dict):
    if report_dict["ece"] < 0.05 and report_dict["mce"] < 0.20:
        return "SUPPORTED"
    elif report_dict["ece"] < 0.10:
        return "PARTIALLY SUPPORTED"
    return "NOT SUPPORTED"

all_verdict_rows = []

# ---------------- SLURP verdicts ----------------
if "slurp" in ACTIVE_DATASETS:
    s = R["slurp"]
    h2_verdict = verdict_h2_from_report(s["h2_report"])
    s["h2_verdict"] = h2_verdict

    aurc = s["aurc_df"].set_index("signal")["aurc"]
    h3_verdict = verdict_h3(aurc, s["bootstrap_df"], "voxintel_r_existing_risk",
                             ["always_execute", "asr_confidence", "intent_confidence"])
    s["h3_verdict"] = h3_verdict

    bt = s["bootstrap_df"].set_index("comparison")
    favors_vs_asr = bool(bt.loc["voxintel_r_existing_risk vs asr_confidence", "favors_risk"]) if \
        "voxintel_r_existing_risk vs asr_confidence" in bt.index else False
    favors_vs_intent = bool(bt.loc["voxintel_r_existing_risk vs intent_confidence", "favors_risk"]) if \
        "voxintel_r_existing_risk vs intent_confidence" in bt.index else False

    h4_verdict = s["h4_verdict"]  # already set to INCONCLUSIVE in Section 16

    verdicts_df = pd.DataFrame([
        {"dataset": "slurp", "hypothesis": "H1", "verdict": "NOT SUPPORTED",
         "note": "Evaluated in Notebook 17 (provisional, FSC-frozen); not rerun here."},
        {"dataset": "slurp", "hypothesis": "H2", "verdict": h2_verdict,
         "note": f"existing risk: ECE={s['h2_report']['ece']:.4f}, MCE={s['h2_report']['mce']:.4f}, "
                 f"Brier={s['h2_report']['brier']:.4f}"},
        {"dataset": "slurp", "hypothesis": "H3", "verdict": h3_verdict,
         "note": f"AURC existing_risk={aurc.loc['voxintel_r_existing_risk']:.6f} vs "
                 f"always_execute={aurc.loc['always_execute']:.6f}, "
                 f"asr_confidence={aurc.loc['asr_confidence']:.6f}, intent_confidence={aurc.loc['intent_confidence']:.6f}; "
                 f"AURC-bootstrap favors risk vs asr={favors_vs_asr}, vs intent={favors_vs_intent}"},
        {"dataset": "slurp", "hypothesis": "H4", "verdict": h4_verdict, "note": s["h4_detail"]},
    ])
    all_verdict_rows.append(verdicts_df)
    print("=== [slurp] verdicts ===")
    print(verdicts_df.drop(columns=["dataset"]).to_string(index=False))

    with open(OUT_DIR / "slurp_voxintel_r_final_summary.json", "w") as f:
        json.dump({
            "dataset": "slurp",
            "h1": {"verdict": "NOT SUPPORTED", "source": "Notebook 17 (provisional)"},
            "h2": {"verdict": h2_verdict, **{k: float(v) for k, v in s["h2_report"].items() if k != "variant"}},
            "h3": {"verdict": h3_verdict, "aurc": {k: float(v) for k, v in aurc.to_dict().items()}},
            "h4": {"verdict": h4_verdict, "detail": s["h4_detail"]},
            "random_seed": RANDOM_SEED,
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }, f, indent=2)
    print(f"Saved: {OUT_DIR / 'slurp_voxintel_r_final_summary.json'}")

# ---------------- FSC verdicts ----------------
if "fsc" in ACTIVE_DATASETS:
    def verdict_h4_fsc(win_counts, severity_is_simulated):
        n_wins = int(win_counts.get("voxintel_r_severity_aware", 0))
        if severity_is_simulated:
            # A win-count against hash-assigned severity is a sensitivity-analysis result, not
            # evidence for or against H4 -- see Section 28.
            return "INCONCLUSIVE", n_wins
        if n_wins >= 5:
            return "SUPPORTED", n_wins
        elif n_wins >= 2:
            return "PARTIALLY SUPPORTED", n_wins
        return "NOT SUPPORTED", n_wins

    h2_verdict = verdict_h2_from_report(R["fsc"]["calibration_metrics_df"]
                                         .set_index(["model", "calibration"])
                                         .loc[("C", R["fsc"]["best_variant_C"])].to_dict())

    fsc_aurc = R["fsc"]["comparison_table"].set_index("signal")["aurc"]
    h3_verdict = verdict_h3(fsc_aurc, R["fsc"]["h3_bootstrap_df"], "voxintel_r_risk_C",
                             ["always_execute", "asr_confidence", "intent_confidence"])
    h4_verdict, h4_n_wins = verdict_h4_fsc(R["fsc"]["win_counts"], R["fsc"]["severity_is_simulated"])
    R["fsc"].update(h2_verdict=h2_verdict, h3_verdict=h3_verdict,
                     h4_verdict=h4_verdict, h4_n_wins=h4_n_wins)

    h2_row = R["fsc"]["calibration_metrics_df"].set_index(["model", "calibration"]).loc[("C", R["fsc"]["best_variant_C"])]

    h3_bt = R["fsc"]["h3_bootstrap_df"].set_index("comparison")
    h3_favors_vs_asr = bool(h3_bt.loc["voxintel_r_risk_C vs asr_confidence", "favors_risk"]) if \
        "voxintel_r_risk_C vs asr_confidence" in h3_bt.index else False
    h3_favors_vs_intent = bool(h3_bt.loc["voxintel_r_risk_C vs intent_confidence", "favors_risk"]) if \
        "voxintel_r_risk_C vs intent_confidence" in h3_bt.index else False
    h3_note = (
        f"AURC voxintel_r_risk_C={fsc_aurc.loc['voxintel_r_risk_C']:.6f} vs "
        f"always_execute={fsc_aurc.loc['always_execute']:.6f}, "
        f"asr_confidence={fsc_aurc.loc['asr_confidence']:.6f}, "
        f"intent_confidence={fsc_aurc.loc['intent_confidence']:.6f}; "
        f"AURC-bootstrap favors risk vs asr={h3_favors_vs_asr}, vs intent={h3_favors_vs_intent} "
        f"(n_test={len(R['fsc']['y_test'])}, {int(R['fsc']['y_test'].sum())} positives -- see Section 27b)"
    )

    h4_note = (
        f"Reported INCONCLUSIVE: cost model uses SIMULATED/hash-assigned severity (Section 28), so this "
        f"is a sensitivity analysis only, not a hypothesis test. Under that simulated cost model, "
        f"voxintel_r_severity_aware achieved the lowest cost in {h4_n_wins}/9 regime x "
        f"clarification-cost combinations -- see fsc_voxintel_r_cost_sensitive_results.csv for detail."
        if R["fsc"]["severity_is_simulated"] else
        f"voxintel_r_severity_aware achieved the lowest cost in {h4_n_wins}/9 regime x clarification-cost "
        f"combinations, under a real (non-simulated) severity mapping."
    )

    verdicts_df = pd.DataFrame([
        {"dataset": "fsc", "hypothesis": "H1", "verdict": "NOT SUPPORTED",
         "note": "Evaluated in Notebook 17; not rerun here."},
        {"dataset": "fsc", "hypothesis": "H2", "verdict": h2_verdict,
         "note": f"model C, {R['fsc']['best_variant_C']} (pre-specified): ECE={h2_row['ece']:.4f}, "
                 f"MCE={h2_row['mce']:.4f}, Brier={h2_row['brier']:.4f}"},
        {"dataset": "fsc", "hypothesis": "H3", "verdict": h3_verdict, "note": h3_note},
        {"dataset": "fsc", "hypothesis": "H4", "verdict": h4_verdict, "note": h4_note},
    ])
    all_verdict_rows.append(verdicts_df)
    print("\n=== [fsc] verdicts ===")
    print(verdicts_df.drop(columns=["dataset"]).to_string(index=False))

    with open(OUT_DIR / "fsc_voxintel_r_final_summary.json", "w") as f:
        json.dump({
            "dataset": "fsc",
            "h1": {"verdict": "NOT SUPPORTED", "source": "Notebook 17"},
            "h2": {"verdict": h2_verdict, "ece": float(h2_row["ece"]), "mce": float(h2_row["mce"]),
                   "brier": float(h2_row["brier"]), "calibration_variant": R["fsc"]["best_variant_C"],
                   "calibration_variant_prespecified": True},
            "h3": {"verdict": h3_verdict,
                   "aurc": {k: float(v) for k, v in fsc_aurc.to_dict().items()}},
            "h4": {"verdict": h4_verdict, "severity_aware_wins_of_9": h4_n_wins,
                   "severity_is_simulated": bool(R["fsc"]["severity_is_simulated"])},
            "random_seed": RANDOM_SEED,
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }, f, indent=2)
    print(f"Saved: {OUT_DIR / 'fsc_voxintel_r_final_summary.json'}")

combined_verdicts_df = pd.concat(all_verdict_rows, ignore_index=True)
combined_verdicts_df.to_csv(OUT_DIR / "voxintel_r_final_verdicts_combined.csv", index=False)
combined_verdicts_df.to_csv(OUT_DIR / "notebook18_hypothesis_results.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'voxintel_r_final_verdicts_combined.csv'}")
print(f"Saved: {OUT_DIR / 'notebook18_hypothesis_results.csv'}")

=== [slurp] verdicts ===
hypothesis       verdict                                                                                                                                                                                                                                                                 note
        H1 NOT SUPPORTED                                                                                                                                                                                                  Evaluated in Notebook 17 (provisional, FSC-frozen); not rerun here.
        H2     SUPPORTED                                                                                                                                                                                                                  existing risk: ECE=0.0192, MCE=0.0841, Brier=0.1025
        H3     SUPPORTED                                                                                             

## 34 — Cross-dataset synthesis: does the methodology generalize?

Compares the already-computed, independently-derived verdicts from Section 33 -- it does **not** re-fit,
re-threshold, or re-evaluate anything, and it does not touch either dataset's rows again. Because SLURP
and FSC do not run identical protocols (Sections 05/08), "generalizes" here means "the verdict direction
agrees where both datasets could actually be evaluated the same way" -- H4 is explicitly a single-dataset
result (FSC only) and is reported as such rather than forced into a two-dataset comparison it cannot
support.

In [40]:
def synthesize(hypothesis, key_getter, treat_as_single_dataset=False):
    if len(ACTIVE_DATASETS) < 2 or treat_as_single_dataset:
        available = {ds: key_getter(ds) for ds in ACTIVE_DATASETS if ds in R and key_getter(ds) is not None}
        return {
            "hypothesis": hypothesis, "status": "SINGLE-DATASET / NOT COMPARABLE",
            "detail": (f"{available} -- " if available else "") +
                      ("SLURP's artifact contract does not support this hypothesis (see Section 16)."
                       if treat_as_single_dataset else
                       f"Only {ACTIVE_DATASETS} available this run."),
        }
    verdicts = {ds: key_getter(ds) for ds in ACTIVE_DATASETS}
    comparable = {ds: v for ds, v in verdicts.items() if v not in (None, "INCONCLUSIVE")}
    if len(comparable) < 2:
        status = "INCONCLUSIVE" if any(v == "INCONCLUSIVE" for v in verdicts.values()) else \
                 "SINGLE-DATASET / NOT COMPARABLE"
        detail_reason = ("at least one dataset's own verdict is INCONCLUSIVE" if status == "INCONCLUSIVE"
                          else "fewer than 2 datasets produced a verdict")
        return {"hypothesis": hypothesis, "status": status,
                "detail": f"verdicts={verdicts} -- {detail_reason}, so agreement cannot be assessed."}
    both_supported_family = all(v in ("SUPPORTED", "PARTIALLY SUPPORTED") for v in comparable.values())
    both_not_supported = all(v == "NOT SUPPORTED" for v in comparable.values())
    status = "CONSISTENT" if (both_supported_family or both_not_supported) else "DIVERGENT"
    detail = ", ".join(f"{ds}={v}" for ds, v in verdicts.items())
    return {"hypothesis": hypothesis, "status": status, "detail": detail}

cross_dataset_rows = [
    synthesize("H2 (calibration)", lambda ds: R.get(ds, {}).get("h2_verdict")),
    synthesize("H3 (selective prediction)", lambda ds: R.get(ds, {}).get("h3_verdict")),
    synthesize("H4 (cost-sensitive decision making)", lambda ds: R.get(ds, {}).get("h4_verdict"),
               treat_as_single_dataset=True),
]
cross_dataset_df = pd.DataFrame(cross_dataset_rows)
print(cross_dataset_df.to_string(index=False))

cross_dataset_df.to_csv(OUT_DIR / "voxintel_r_cross_dataset_summary.csv", index=False)
cross_dataset_df.to_csv(OUT_DIR / "notebook18_cross_dataset_summary.csv", index=False)

cross_dataset_summary = {
    "active_datasets": ACTIVE_DATASETS,
    "skipped_datasets": {ds: err for ds, err in dataset_load_errors.items()},
    "cross_dataset_verdicts": cross_dataset_rows,
    "protocol_note": ("SLURP is an existing evaluation population (no independent split); FSC has a "
                       "genuine validation/test split. H4 is FSC-only by design -- see Section 16."),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
with open(OUT_DIR / "voxintel_r_cross_dataset_synthesis.json", "w") as f:
    json.dump(cross_dataset_summary, f, indent=2, default=str)

print(f"\nSaved: {OUT_DIR / 'voxintel_r_cross_dataset_summary.csv'}")
print(f"Saved: {OUT_DIR / 'notebook18_cross_dataset_summary.csv'}")
print(f"Saved: {OUT_DIR / 'voxintel_r_cross_dataset_synthesis.json'}")

if len(ACTIVE_DATASETS) < 2:
    print("\nNOTE: cross-dataset comparison is incomplete this run -- see Section 10 for why the "
          "missing dataset's artifact could not be loaded.")

                         hypothesis                          status                                                                                                                           detail
                   H2 (calibration)                      CONSISTENT                                                                                         slurp=SUPPORTED, fsc=PARTIALLY SUPPORTED
          H3 (selective prediction)                      CONSISTENT                                                                                         slurp=SUPPORTED, fsc=PARTIALLY SUPPORTED
H4 (cost-sensitive decision making) SINGLE-DATASET / NOT COMPARABLE {'slurp': 'INCONCLUSIVE', 'fsc': 'INCONCLUSIVE'} -- SLURP's artifact contract does not support this hypothesis (see Section 16).

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\voxintel_r_cross_dataset_summary.csv
Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\notebook18_cross_dataset_summary.csv
Saved: C:\Users\ACER\OneDri

## 35 — Limitations

- **Small positive class on FSC.** The FSC test split has only 23 downstream intent failures out of 3,793
  utterances (0.6064%); SLURP's positive-class rate is much higher (≈21.3%, see Section 13/18). Every
  calibration, selective-prediction, and cost-sensitivity number on FSC should be read with that in mind
  -- confidence intervals in Section 32 are wide.
- **SLURP has no independent held-out split.** Every SLURP number in this notebook is an evaluation of an
  *existing, already-frozen* risk score against its deployment population -- it is not a train/test
  generalization estimate in the way the FSC numbers are. This is a genuine, structural limitation of the
  current SLURP artifact, not a shortcut taken by this notebook; Section 08/16 make explicit which SLURP
  analyses are simply not possible without a different upstream artifact (a SLURP validation split, or a
  SLURP artifact that separately supports fitting and holding out).
- **H4 could not be evaluated on SLURP at all.** This is reported as `INCONCLUSIVE`, not folded into a
  cross-dataset average or silently dropped.
- **Simulated severity tiers.** Neither dataset has real severity ground truth; every cost-sensitivity
  number (FSC only) is a sensitivity analysis over *assumed* cost regimes, not a real-world cost estimate.
  Edit `SEVERITY_OVERRIDES` in Section 28 with a real taxonomy mapping if one becomes available.
- **Random Forest only.** Section 04 notes Random Forest was selected over logistic regression in
  Notebook 17; this notebook does not re-run that model-family comparison.
- **Single train/refit per feature-group/ablation subset.** Random Forest has some fit-to-fit variance;
  Section 19's 5-fold CV gives a variance estimate on validation, but the single frozen model evaluated on
  test is one draw from that variance, for FSC. There is no equivalent variance estimate for SLURP's
  existing risk, since it was not fit in this notebook.
- **Artifact-column assumptions are now explicit, not implicit.** Section 09/10 name every column this
  notebook expects and where it looks for it; if the upstream artifact's schema changes, Section 10 fails
  loudly with the exact missing column rather than this notebook silently computing something else.

## 36 — Final research conclusions

**The research story, in technical terms:**

VoxIntel began as a diagnostic study of how ASR errors propagate into downstream intent failures.
Historical notebooks established that ASR errors do propagate into intent failures, that WER alone does
not capture semantic risk, and that reference-aware features can predict failure well offline -- but
none of that is usable by a deployed system that does not have the reference transcript at inference
time. VoxIntel-R restricted itself to inference-time-only signals and Notebook 17 tested **H1**: whether
ASR-native uncertainty adds anything beyond intent-native uncertainty. It did not (Section 06).

This notebook picked up the three hypotheses that remain after H1 failed -- **H2** (is the risk
calibrated?), **H3** (does it improve selective prediction over naive baselines?), and **H4** (does a
cost-aware policy built on it beat uniform thresholds?) -- and tested each **under the artifact contract
each dataset actually supports**: SLURP's existing risk was evaluated as-is, with no new fitting; FSC's
risk was fit on validation, frozen, and evaluated once on test.

**Read the live verdict table in Section 33/34 above for this run's actual results** -- printed values
reflect whatever is in the current `fsc_voxintel_r_features.csv` / `voxintel_r_predictions.csv`, not
numbers baked into this notebook's prose.

**What this version of the notebook fixed relative to the first version:** the first version could not
actually get past the per-dataset risk-modeling step for SLURP, because it assumed `R["slurp"]["y_val"]`
and `R["slurp"]["val_scores"]` existed (they cannot, by SLURP's own artifact contract) and assumed
`fsc_voxintel_r_features.csv` already contained a `voxintel_r_risk` column (it does not -- that risk is
fit *by* this notebook, in Section 19). This version makes every one of those distinctions an explicit,
checked branch instead of an assumption. **Revision v3 then fixed four methodological issues found by
review of the v2 rewrite** (out-of-fold calibration fitting, test-blind calibration-variant selection, an
honest INCONCLUSIVE default for H4 under simulated severity, and AURC-based rather than ROC-AUC-based
significance testing for SLURP's H3). **This revision (v4) fixed three further issues found by review of
v3's executed output**: FSC's calibration method is now pre-specified as sigmoid rather than picked by
validation ECE, because that selection step could still be won by isotonic overfitting its own
out-of-fold fitting data (Section 23); FSC's H3 verdict now requires a bootstrap-significant win, not
just a point-estimate one, closing the gap where a ~0.00002 AURC margin on 23 test positives could be
reported as `SUPPORTED` (Section 27b/33); and H2's ECE-vs-MCE decision hierarchy is now stated explicitly
in Section 33. See the changelog at the top of Section 00 for the full list, with the specific numbers
that motivated each fix.

## 37 — This run's summary

In [41]:
from IPython.display import display, Markdown

lines = ["**This run's per-dataset verdicts:**", ""]
for ds in ACTIVE_DATASETS:
    r = R[ds]
    lines.append(
        f"- **{DATASET_CONFIG[ds]['label']}** ({r['evaluation_mode']}): "
        f"H2 (calibration) = **{r.get('h2_verdict', 'n/a')}**, "
        f"H3 (selective prediction) = **{r.get('h3_verdict', 'n/a')}**, "
        f"H4 (cost-sensitive decision) = **{r.get('h4_verdict', 'n/a')}**"
    )
lines.append("")
lines.append("See Section 34 for the cross-dataset synthesis.")
display(Markdown("\n".join(lines)))

**This run's per-dataset verdicts:**

- **SLURP** (existing_evaluation_population): H2 (calibration) = **SUPPORTED**, H3 (selective prediction) = **SUPPORTED**, H4 (cost-sensitive decision) = **INCONCLUSIVE**
- **FSC** (split): H2 (calibration) = **PARTIALLY SUPPORTED**, H3 (selective prediction) = **PARTIALLY SUPPORTED**, H4 (cost-sensitive decision) = **INCONCLUSIVE**

See Section 34 for the cross-dataset synthesis.

## 38 — Consolidated Notebook 18 output files

The sections above save results under their established per-dataset project naming convention (e.g.
`fsc_voxintel_r_final_calibration_metrics.csv`, `slurp_voxintel_r_evaluation_summary.csv`). This section
additionally writes the consolidated `notebook18_*` files (most were already written inline above;
this cell is the single place that both writes the remaining ones and lists all of them together for
convenience). `notebook18_audit_log.csv` is written here because `AUDIT_LOG` is only complete once every
section above has run.

In [42]:
audit_log_df = pd.DataFrame(AUDIT_LOG)
audit_log_df.to_csv(OUT_DIR / "notebook18_audit_log.csv", index=False)

notebook18_files = [
    "notebook18_artifact_audit.csv",       # Section 10b
    "notebook18_slurp_evaluation.csv",     # Section 18
    "notebook18_fsc_cv_results.csv",       # Section 19
    "notebook18_fsc_test_results.csv",     # Section 27
    "notebook18_calibration_results.csv",  # Section 23 (FSC)
    "notebook18_reliability_data.csv",     # Section 23 (FSC)
    "notebook18_hypothesis_results.csv",   # Section 33
    "notebook18_cross_dataset_summary.csv",# Section 34
    "notebook18_audit_log.csv",            # this cell
]

print("=== Notebook 18 consolidated output files ===")
for fname in notebook18_files:
    p = OUT_DIR / fname
    status = f"{p.stat().st_size} bytes" if p.exists() else "MISSING"
    print(f"  {fname:45s} {status}")

=== Notebook 18 consolidated output files ===
  notebook18_artifact_audit.csv                 578 bytes
  notebook18_slurp_evaluation.csv               496 bytes
  notebook18_fsc_cv_results.csv                 183 bytes
  notebook18_fsc_test_results.csv               526 bytes
  notebook18_calibration_results.csv            491 bytes
  notebook18_reliability_data.csv               3220 bytes
  notebook18_hypothesis_results.csv             1450 bytes
  notebook18_cross_dataset_summary.csv          379 bytes
  notebook18_audit_log.csv                      8387 bytes


## 39 — Artifact manifest

Every artifact this notebook wrote, with existence and size checked directly against disk rather than
assumed -- covering both datasets' per-dataset files, the SLURP evaluation-only files, and the
combined/cross-dataset/consolidated files.

In [43]:
expected_artifacts = []

if "slurp" in ACTIVE_DATASETS:
    expected_artifacts += [
        OUT_DIR / "slurp_voxintel_r_aurc.csv",
        OUT_DIR / "slurp_voxintel_r_rank_metrics.csv",
        OUT_DIR / "slurp_voxintel_r_descriptive_operating_points.csv",
        OUT_DIR / "slurp_voxintel_r_bootstrap_h3.csv",
        OUT_DIR / "slurp_voxintel_r_evaluation_summary.csv",
        OUT_DIR / "slurp_voxintel_r_final_summary.json",
    ]
    if R["slurp"].get("subgroup_df") is not None:
        expected_artifacts.append(OUT_DIR / "slurp_voxintel_r_intent_subgroup_failure_rates.csv")
    if R["slurp"].get("group_comparison_df") is not None:
        expected_artifacts.append(OUT_DIR / "slurp_voxintel_r_group_comparison.csv")

if "fsc" in ACTIVE_DATASETS:
    expected_artifacts += [
        OUT_DIR / "fsc_voxintel_r_cv_results.csv",
        OUT_DIR / "fsc_voxintel_r_calibration_diagnostic_validation.csv",
        OUT_DIR / "fsc_voxintel_r_final_calibration_metrics.csv",
        OUT_DIR / "fsc_voxintel_r_reliability_data.csv",
        OUT_DIR / "fsc_voxintel_r_reliability_diagram.png",
        OUT_DIR / "fsc_voxintel_r_risk_coverage.csv",
        OUT_DIR / "fsc_voxintel_r_risk_coverage.png",
        OUT_DIR / "fsc_voxintel_r_selective_results.csv",
        OUT_DIR / "fsc_voxintel_r_bootstrap_h3.csv",
        OUT_DIR / "fsc_voxintel_r_cost_sensitive_results.csv",
        OUT_DIR / "fsc_voxintel_r_cost_sensitivity.csv",
        OUT_DIR / "fsc_voxintel_r_cost_sensitivity.png",
        OUT_DIR / "fsc_voxintel_r_feature_ablation.csv",
        OUT_DIR / "fsc_voxintel_r_bootstrap_ablation.csv",
        OUT_DIR / "fsc_voxintel_r_final_summary.json",
    ]

expected_artifacts += [
    OUT_DIR / "notebook18_artifact_audit.csv",
    OUT_DIR / "notebook18_slurp_evaluation.csv",
    OUT_DIR / "notebook18_fsc_cv_results.csv",
    OUT_DIR / "notebook18_fsc_test_results.csv",
    OUT_DIR / "notebook18_calibration_results.csv",
    OUT_DIR / "notebook18_reliability_data.csv",
    OUT_DIR / "notebook18_hypothesis_results.csv",
    OUT_DIR / "notebook18_cross_dataset_summary.csv",
    OUT_DIR / "notebook18_audit_log.csv",
    OUT_DIR / "voxintel_r_final_verdicts_combined.csv",
    OUT_DIR / "voxintel_r_cross_dataset_summary.csv",
    OUT_DIR / "voxintel_r_cross_dataset_synthesis.json",
]

manifest_rows = []
for p in expected_artifacts:
    manifest_rows.append({
        "path": str(p),
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else None,
    })

manifest_df = pd.DataFrame(manifest_rows)
print(manifest_df.to_string(index=False))

n_missing = (~manifest_df["exists"]).sum()
if n_missing > 0:
    print(f"\nWARNING: {n_missing} expected artifact(s) missing -- see table above.")
else:
    print(f"\nAll {len(manifest_df)} expected artifacts present in {OUT_DIR}.")

manifest_df.to_csv(OUT_DIR / "notebook18_artifact_manifest.csv", index=False)
print(f"Saved: {OUT_DIR / 'notebook18_artifact_manifest.csv'}")

                                                                                                path  exists  size_bytes
                           C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_aurc.csv    True         199
                   C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_rank_metrics.csv    True         251
   C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_descriptive_operating_points.csv    True         469
                   C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_bootstrap_h3.csv    True         435
             C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_evaluation_summary.csv    True         496
                 C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_final_summary.json    True         931
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\slurp_voxintel_r_intent_subgroup_failure_rates.csv    True        2563
                       C:\Users\